# F — Ablation and Stability Evidence

This section contains the tuned-only ablation, stability, and reviewer-facing robustness layers that feed the manuscript appendix and the final paper freeze.

# F1 — Define Ablation Study Design and Protocol

In [ ]:
# ==============================================================
# F1 — Define Ablation Study Design and Protocol
# --------------------------------------------------------------
# Purpose:
#   Define the ablation study configuration, feature-block logic,
#   evaluation protocol, and documentation layer for the benchmark.
#
# Methodological note:
#   This step does not train any model. It only formalizes the
#   ablation study that will be executed in subsequent steps.
#
#   The ablation study is restricted to the tuned models only, so
#   that each family is evaluated in its strongest benchmark form.
#
#   Models included:
#     - linear_tuned
#     - neural_tuned
#     - cox_tuned
#     - deepsurv_tuned
#
#   Main goal:
#     quantify how much performance depends on different feature
#     blocks, especially:
#       - static covariates
#       - early-window behavior summaries
#       - dynamic person-period temporal signals
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - HORIZONS_WEEKS
#
# Outputs:
#   - ablation_config.json
#   - table_ablation_model_registry.csv
#   - table_ablation_feature_blocks.csv
#   - table_ablation_scenarios.csv
# ==============================================================

print("\n" + "=" * 70)
print("F1 — Define Ablation Study Design and Protocol")
print("=" * 70)
print("Methodological note: this step defines the ablation study only.")
print("No model is trained here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json", "HORIZONS_WEEKS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd

# ------------------------------
# 2) Define tuned models included in ablation
# ------------------------------
ABLATION_MODEL_REGISTRY = [
    {
        "model_id": "linear_tuned",
        "display_name": "Linear Discrete-Time Hazard (Tuned)",
        "family": "discrete_time_linear",
        "base_data_level": "person_period",
        "uses_dynamic_temporal_features": True,
        "uses_static_features": True,
        "uses_early_window_features": False,
        "ablation_positioning_note": "Tuned representative of the linear discrete-time family.",
    },
    {
        "model_id": "neural_tuned",
        "display_name": "Neural Discrete-Time Survival (Tuned)",
        "family": "discrete_time_neural",
        "base_data_level": "person_period",
        "uses_dynamic_temporal_features": True,
        "uses_static_features": True,
        "uses_early_window_features": False,
        "ablation_positioning_note": "Tuned representative of the neural discrete-time family.",
    },
    {
        "model_id": "cox_tuned",
        "display_name": "Cox Comparable (Tuned)",
        "family": "continuous_time_cox",
        "base_data_level": "enrollment",
        "uses_dynamic_temporal_features": False,
        "uses_static_features": True,
        "uses_early_window_features": True,
        "ablation_positioning_note": "Tuned representative of the Cox comparable family.",
    },
    {
        "model_id": "deepsurv_tuned",
        "display_name": "DeepSurv (Tuned)",
        "family": "continuous_time_deepsurv",
        "base_data_level": "enrollment",
        "uses_dynamic_temporal_features": False,
        "uses_static_features": True,
        "uses_early_window_features": True,
        "ablation_positioning_note": "Tuned representative of the DeepSurv family.",
    },
]

ablation_model_registry_df = pd.DataFrame(ABLATION_MODEL_REGISTRY)

# ------------------------------
# 3) Define feature blocks
# ------------------------------
ABLATION_FEATURE_BLOCKS = [
    {
        "block_id": "static_structural",
        "block_label": "Static structural covariates",
        "applies_to_data_level": "both",
        "conceptual_role": "student background and enrollment-level structure",
        "examples": "gender, region, highest_education, imd_band, age_band, num_of_prev_attempts, studied_credits, disability",
    },
    {
        "block_id": "dynamic_temporal",
        "block_label": "Dynamic temporal person-period signals",
        "applies_to_data_level": "person_period",
        "conceptual_role": "time-varying weekly engagement and progression",
        "examples": "week, total_clicks_week, active_this_week, n_vle_rows_week, n_distinct_sites_week, cum_clicks_until_t, recency, streak",
    },
    {
        "block_id": "early_window_behavior",
        "block_label": "Early-window behavior summaries",
        "applies_to_data_level": "enrollment",
        "conceptual_role": "compressed early-course activity profile",
        "examples": "clicks_first_4_weeks, active_weeks_first_4, mean_clicks_first_4_weeks",
    },
]

ablation_feature_blocks_df = pd.DataFrame(ABLATION_FEATURE_BLOCKS)

# ------------------------------
# 4) Define ablation scenarios
# ------------------------------
# Strategy:
#   For each family, compare:
#   - full tuned model
#   - remove structural/static block
#   - remove temporal/early-window block
#   - only structural/static block
#   - only temporal/early-window block
#
# Note:
#   The exact executable datasets will be materialized later.
ABLATION_SCENARIOS = [
    {
        "scenario_id": "full_features",
        "scenario_label": "Full tuned feature set",
        "scenario_type": "reference",
        "interpretation_goal": "Reference tuned benchmark for each family.",
    },
    {
        "scenario_id": "drop_static_structural",
        "scenario_label": "Drop static structural covariates",
        "scenario_type": "leave_one_block_out",
        "interpretation_goal": "Estimate how much performance depends on background/structural covariates.",
    },
    {
        "scenario_id": "drop_temporal_signal",
        "scenario_label": "Drop temporal signal block",
        "scenario_type": "leave_one_block_out",
        "interpretation_goal": "Estimate how much performance depends on behavioral engagement dynamics.",
    },
    {
        "scenario_id": "only_static_structural",
        "scenario_label": "Only static structural covariates",
        "scenario_type": "single_block_only",
        "interpretation_goal": "Assess how far structural covariates alone can go.",
    },
    {
        "scenario_id": "only_temporal_signal",
        "scenario_label": "Only temporal signal block",
        "scenario_type": "single_block_only",
        "interpretation_goal": "Assess how far behavior/activity signals alone can go.",
    },
]

ablation_scenarios_df = pd.DataFrame(ABLATION_SCENARIOS)

# ------------------------------
# 5) Protocol definition
# ------------------------------
ABLATION_PROTOCOL = {
    "ablation_scope": "tuned_models_only",
    "included_models": [item["model_id"] for item in ABLATION_MODEL_REGISTRY],
    "main_question": (
        "How much of model performance is driven by structural covariates versus temporal/behavioral information?"
    ),
    "secondary_question": (
        "Does the relative importance of feature blocks vary across model families?"
    ),
    "evaluation_protocol": {
        "reuse_existing_train_test_split": True,
        "reuse_existing_horizons": [int(h) for h in HORIZONS_WEEKS],
        "reuse_existing_primary_metrics": [
            "ibs",
            "c_index",
            "brier_at_horizon",
            "calibration_at_horizon",
            "risk_auc_at_horizon",
        ],
        "primary_comparison_logic": (
            "Compare each ablation scenario against the full tuned version within the same model family."
        ),
    },
    "interpretation_rules": {
        "large_drop_after_removal": "The removed block is important for that family.",
        "small_drop_after_removal": "The removed block contributes little incremental signal.",
        "strong_only_block_performance": "That block alone carries substantial predictive information.",
        "cross_family_difference": (
            "Different model families may exploit the same information blocks differently."
        ),
    },
    "paper_positioning_note": (
        "The ablation study is intended to explain performance sources after the benchmark ranking is already established."
    ),
}

# ------------------------------
# 6) Save outputs
# ------------------------------
model_registry_path = TABLES_DIR / "table_ablation_model_registry.csv"
feature_blocks_path = TABLES_DIR / "table_ablation_feature_blocks.csv"
scenarios_path = TABLES_DIR / "table_ablation_scenarios.csv"
config_path = METADATA_DIR / "ablation_config.json"

ablation_model_registry_df.to_csv(model_registry_path, index=False)
ablation_feature_blocks_df.to_csv(feature_blocks_path, index=False)
ablation_scenarios_df.to_csv(scenarios_path, index=False)
save_json(ABLATION_PROTOCOL, config_path)

# ------------------------------
# 7) Output for feedback
# ------------------------------
print("\nAblation model registry:")
display(ablation_model_registry_df)

print("\nAblation feature blocks:")
display(ablation_feature_blocks_df)

print("\nAblation scenarios:")
display(ablation_scenarios_df)

print("\nAblation protocol summary:")
display(pd.DataFrame([{
    "ablation_scope": ABLATION_PROTOCOL["ablation_scope"],
    "included_models": ", ".join(ABLATION_PROTOCOL["included_models"]),
    "reuse_existing_train_test_split": ABLATION_PROTOCOL["evaluation_protocol"]["reuse_existing_train_test_split"],
    "reuse_existing_horizons": ", ".join(str(x) for x in ABLATION_PROTOCOL["evaluation_protocol"]["reuse_existing_horizons"]),
    "main_question": ABLATION_PROTOCOL["main_question"],
    "secondary_question": ABLATION_PROTOCOL["secondary_question"],
}]))

print("\nSaved:")
print("-", model_registry_path.resolve())
print("-", feature_blocks_path.resolve())
print("-", scenarios_path.resolve())
print("-", config_path.resolve())

# F2 — Materialize Ablation Variants (Revised v2)

### Funcao: load_ready_dataset

Definicao isolada de load_ready_dataset para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# F2 — Materialize Ablation Variants (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Materialize ablation-ready dataset variants for the tuned
#   benchmark models, without training them yet.
#
# Methodological note:
#   This revised version corrects the discrete-time ablation logic:
#
#   - for discrete-time models, "week" is treated as a structural
#     time index and is ALWAYS retained;
#   - the ablative temporal block for discrete-time families
#     therefore excludes "week" and only includes behavioral
#     temporal signals.
#
#   This preserves the temporal hazard structure required for
#   discrete-time survival reconstruction.
# ==============================================================

print("\n" + "=" * 70)
print("F2 — Materialize Ablation Variants (Revised v2)")
print("=" * 70)
print("Methodological note: this step materializes ablation-ready")
print("feature subsets only. No model is trained here.")
print("Revised behavior: ready datasets are loaded from disk.")
print("Important correction: in discrete-time families, 'week' is")
print("always retained as a structural time index.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import pandas as pd

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Helper to load dataset
# ------------------------------
def load_ready_dataset(base_name: str) -> pd.DataFrame:
    parquet_path = DATA_OUTPUT_DIR / f"{base_name}.parquet"
    csv_path = DATA_OUTPUT_DIR / f"{base_name}.csv"

    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)

    raise FileNotFoundError(
        f"Could not find dataset for '{base_name}'. "
        f"Checked:\n- {parquet_path}\n- {csv_path}"
    )

# ------------------------------
# 3) Define feature blocks
# ------------------------------
STATIC_STRUCTURAL_FEATURES = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
]

# For discrete-time models, "week" is NOT ablated away.
DISCRETE_TIME_INDEX_FEATURES = [
    "week",
]

DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES = [
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]

EARLY_WINDOW_FEATURES = [
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]

# ------------------------------
# 4) Define auxiliary / target columns
# ------------------------------
AUX_DISCRETE = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "event_observed",
    "t_event_week",
    "t_final_week",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_DISCRETE = ["event_t"]

AUX_ENROLLMENT = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "duration",
    "duration_raw",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_ENROLLMENT = ["event"]

# ------------------------------
# 5) Scenario map by family
# ------------------------------
SCENARIO_MAP = {
    "full_features": {
        "discrete_time": STATIC_STRUCTURAL_FEATURES + DISCRETE_TIME_INDEX_FEATURES + DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES,
        "enrollment": STATIC_STRUCTURAL_FEATURES + EARLY_WINDOW_FEATURES,
    },
    "drop_static_structural": {
        "discrete_time": DISCRETE_TIME_INDEX_FEATURES + DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES,
        "enrollment": EARLY_WINDOW_FEATURES,
    },
    "drop_temporal_signal": {
        "discrete_time": STATIC_STRUCTURAL_FEATURES + DISCRETE_TIME_INDEX_FEATURES,
        "enrollment": STATIC_STRUCTURAL_FEATURES,
    },
    "only_static_structural": {
        "discrete_time": STATIC_STRUCTURAL_FEATURES + DISCRETE_TIME_INDEX_FEATURES,
        "enrollment": STATIC_STRUCTURAL_FEATURES,
    },
    "only_temporal_signal": {
        "discrete_time": DISCRETE_TIME_INDEX_FEATURES + DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES,
        "enrollment": EARLY_WINDOW_FEATURES,
    },
}

SCENARIO_LABELS = {
    "full_features": "Full tuned feature set",
    "drop_static_structural": "Drop static structural covariates",
    "drop_temporal_signal": "Drop temporal signal block",
    "only_static_structural": "Only static structural covariates",
    "only_temporal_signal": "Only temporal signal block",
}

# ------------------------------
# 6) Dataset registry loaded from disk
# ------------------------------
DATASET_REGISTRY = [
    {
        "model_id": "linear_tuned",
        "family": "discrete_time_linear",
        "data_level": "discrete_time",
        "train_df": load_ready_dataset("pp_linear_hazard_ready_train"),
        "test_df": load_ready_dataset("pp_linear_hazard_ready_test"),
        "aux_cols": AUX_DISCRETE,
        "target_cols": TARGET_DISCRETE,
    },
    {
        "model_id": "neural_tuned",
        "family": "discrete_time_neural",
        "data_level": "discrete_time",
        "train_df": load_ready_dataset("pp_neural_hazard_ready_train"),
        "test_df": load_ready_dataset("pp_neural_hazard_ready_test"),
        "aux_cols": AUX_DISCRETE,
        "target_cols": TARGET_DISCRETE,
    },
    {
        "model_id": "cox_tuned",
        "family": "continuous_time_cox",
        "data_level": "enrollment",
        "train_df": load_ready_dataset("enrollment_cox_ready_train"),
        "test_df": load_ready_dataset("enrollment_cox_ready_test"),
        "aux_cols": AUX_ENROLLMENT,
        "target_cols": TARGET_ENROLLMENT,
    },
    {
        "model_id": "deepsurv_tuned",
        "family": "continuous_time_deepsurv",
        "data_level": "enrollment",
        "train_df": load_ready_dataset("enrollment_deepsurv_ready_train"),
        "test_df": load_ready_dataset("enrollment_deepsurv_ready_test"),
        "aux_cols": AUX_ENROLLMENT,
        "target_cols": TARGET_ENROLLMENT,
    },
]

# ------------------------------
# 7) Materialize variants
# ------------------------------
variant_registry_rows = []
variant_feature_manifest_rows = []

for item in DATASET_REGISTRY:
    model_id = item["model_id"]
    family = item["family"]
    data_level = item["data_level"]
    train_df = item["train_df"]
    test_df = item["test_df"]
    aux_cols = [c for c in item["aux_cols"] if c in train_df.columns]
    target_cols = [c for c in item["target_cols"] if c in train_df.columns]

    for scenario_id, scenario_def in SCENARIO_MAP.items():
        feature_cols = [c for c in scenario_def[data_level] if c in train_df.columns]

        ordered_cols = aux_cols + target_cols + feature_cols
        ordered_cols = [c for c in ordered_cols if c in train_df.columns]

        train_variant = train_df[ordered_cols].copy()
        test_variant = test_df[ordered_cols].copy()

        variant_base_name = f"{model_id}__{scenario_id}"
        train_name = f"{variant_base_name}__train"
        test_name = f"{variant_base_name}__test"

        train_csv = DATA_OUTPUT_DIR / f"{train_name}.csv"
        train_parquet = DATA_OUTPUT_DIR / f"{train_name}.parquet"
        test_csv = DATA_OUTPUT_DIR / f"{test_name}.csv"
        test_parquet = DATA_OUTPUT_DIR / f"{test_name}.parquet"

        train_variant.to_csv(train_csv, index=False)
        test_variant.to_csv(test_csv, index=False)
        train_variant.to_parquet(train_parquet, index=False)
        test_variant.to_parquet(test_parquet, index=False)

        variant_registry_rows.append({
            "model_id": model_id,
            "family": family,
            "data_level": data_level,
            "scenario_id": scenario_id,
            "scenario_label": SCENARIO_LABELS[scenario_id],
            "train_table_name": train_name,
            "test_table_name": test_name,
            "n_train_rows": int(train_variant.shape[0]),
            "n_test_rows": int(test_variant.shape[0]),
            "n_columns_total": int(len(ordered_cols)),
            "n_aux_columns": int(len(aux_cols)),
            "n_target_columns": int(len(target_cols)),
            "n_feature_columns": int(len(feature_cols)),
            "train_csv_path": str(train_csv),
            "train_parquet_path": str(train_parquet),
            "test_csv_path": str(test_csv),
            "test_parquet_path": str(test_parquet),
        })

        for col in ordered_cols:
            if col in aux_cols:
                role = "auxiliary"
            elif col in target_cols:
                role = "target"
            else:
                role = "feature"

            if col in STATIC_STRUCTURAL_FEATURES:
                block = "static_structural"
            elif col in DISCRETE_TIME_INDEX_FEATURES:
                block = "discrete_time_index"
            elif col in DYNAMIC_TEMPORAL_BEHAVIORAL_FEATURES:
                block = "dynamic_temporal_behavioral"
            elif col in EARLY_WINDOW_FEATURES:
                block = "early_window_behavior"
            else:
                block = "aux_or_target"

            variant_feature_manifest_rows.append({
                "model_id": model_id,
                "family": family,
                "data_level": data_level,
                "scenario_id": scenario_id,
                "scenario_label": SCENARIO_LABELS[scenario_id],
                "column_name": col,
                "role": role,
                "feature_block": block,
            })

variant_registry_df = pd.DataFrame(variant_registry_rows)
variant_feature_manifest_df = pd.DataFrame(variant_feature_manifest_rows)

# ------------------------------
# 8) Save registry artifacts
# ------------------------------
variant_registry_path = TABLES_DIR / "table_ablation_variant_registry.csv"
variant_feature_manifest_path = TABLES_DIR / "table_ablation_variant_feature_manifest.csv"
variant_registry_json_path = METADATA_DIR / "ablation_variant_registry.json"

variant_registry_df.to_csv(variant_registry_path, index=False)
variant_feature_manifest_df.to_csv(variant_feature_manifest_path, index=False)

save_json(
    {
        "n_variants_materialized": int(variant_registry_df.shape[0]),
        "models_included": sorted(variant_registry_df["model_id"].unique().tolist()),
        "scenarios_included": sorted(variant_registry_df["scenario_id"].unique().tolist()),
        "data_levels": sorted(variant_registry_df["data_level"].unique().tolist()),
        "discrete_time_week_always_retained": True,
    },
    variant_registry_json_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation variant registry:")
display(variant_registry_df)

print("\nAblation variant feature manifest (first rows):")
display(variant_feature_manifest_df.head(60))

print("\nSaved:")
print("-", variant_registry_path.resolve())
print("-", variant_feature_manifest_path.resolve())
print("-", variant_registry_json_path.resolve())

# F3 — Train and Evaluate Ablation Variants for Linear and Neural Tuned Models (Revised)

In [ ]:
# ==============================================================
# F3 — Train and Evaluate Ablation Variants for Linear and Neural Tuned Models (Revised)
# --------------------------------------------------------------
# Purpose:
#   Train and evaluate the ablation variants for the tuned
#   discrete-time model families:
#     - linear_tuned
#     - neural_tuned
#
# Methodological note:
#   This revised version is compatible with F2 Revised v2:
#   - "week" is always retained for discrete-time families as a
#     structural time index;
#   - the ablative temporal block refers only to behavioral
#     temporal signals, not to the time index itself.
# ==============================================================

print("\n" + "=" * 70)
print("F3 — Train and Evaluate Ablation Variants for Linear and Neural Tuned Models (Revised)")
print("=" * 70)
print("Methodological note: this step trains and evaluates ablation")
print("variants for the tuned discrete-time families only.")
print("Revised behavior: 'week' is always retained as a structural time index.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P29.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Registry / paths
# ------------------------------
DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
ABLATION_REGISTRY_PATH = TABLES_DIR / "table_ablation_variant_registry.csv"

if not ABLATION_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing ablation registry: {ABLATION_REGISTRY_PATH}")

ablation_registry = pd.read_csv(ABLATION_REGISTRY_PATH)

TARGET_MODELS = ["linear_tuned", "neural_tuned"]

ablation_registry_dt = ablation_registry[
    ablation_registry["model_id"].isin(TARGET_MODELS)
].copy()

# ------------------------------
# 4) Column definitions
# ------------------------------
AUX_DISCRETE = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "event_observed",
    "t_event_week",
    "t_final_week",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_COL = "event_t"

CATEGORICAL_FEATURES = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]


### Funcao: load_variant

Definicao isolada de load_variant para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 5) Helpers
# ------------------------------
def load_variant(path_csv: str, path_parquet: str) -> pd.DataFrame:
    p_parquet = Path(path_parquet)
    p_csv = Path(path_csv)
    if p_parquet.exists():
        return pd.read_parquet(p_parquet)
    if p_csv.exists():
        return pd.read_csv(p_csv)
    raise FileNotFoundError(f"Variant file not found:\n- {p_parquet}\n- {p_csv}")


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_DISCRETE + [TARGET_COL])
    return [c for c in df.columns if c not in excluded]


### Funcao: split_feature_types

Definicao isolada de split_feature_types para reutilizacao nas etapas seguintes.


In [ ]:

def split_feature_types(feature_cols):
    categorical_cols = [c for c in feature_cols if c in CATEGORICAL_FEATURES]
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]
    return numeric_cols, categorical_cols


### Funcao: build_preprocessor

Definicao isolada de build_preprocessor para reutilizacao nas etapas seguintes.


In [ ]:

def build_preprocessor(feature_cols):
    numeric_cols, categorical_cols = split_feature_types(feature_cols)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor, numeric_cols, categorical_cols


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_discrete_survival

Definicao isolada de evaluate_discrete_survival para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_discrete_survival(test_pred_df: pd.DataFrame, horizons: list[int]):
    required_cols = ["enrollment_id", "week", "pred_hazard", "event_observed", "time_for_split"]
    missing_cols = [c for c in required_cols if c not in test_pred_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns for discrete survival evaluation: {missing_cols}")

    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )
    test_pred_df["pred_risk"] = 1.0 - test_pred_df["pred_survival"]

    truth_test = (
        test_pred_df.groupby("enrollment_id", as_index=False)
        .agg(
            event=("event_observed", "max"),
            duration=("time_for_split", "max"),
        )
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )

    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)

    surv_df = surv_wide.copy()
    surv_df.columns.name = "enrollment_id"

    durations_test = truth_test["duration"].astype(int).to_numpy()
    events_test = truth_test["event"].astype(int).to_numpy()

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    primary_rows = []
    try:
        c_index = float(eval_surv.concordance_td())
        c_index_note = "pycox_concordance_td"
    except Exception as e:
        c_index = np.nan
        c_index_note = f"failed: {str(e)}"

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
        ibs_note = "pycox_integrated_brier_score"
    except Exception as e:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)}"

    primary_rows.append({"metric_name": "ibs", "metric_value": ibs_value, "notes": ibs_note})
    primary_rows.append({"metric_name": "c_index", "metric_value": c_index, "notes": c_index_note})
    primary_df = pd.DataFrame(primary_rows)

    try:
        brier_h = eval_surv.brier_score(np.array(horizons, dtype=int))
        brier_df = pd.DataFrame({
            "horizon_week": list(brier_h.index.astype(int)),
            "metric_name": ["brier_at_horizon"] * len(brier_h.index),
            "metric_value": list(brier_h.values.astype(float)),
            "notes": ["pycox_brier_score"] * len(brier_h.index),
        })
    except Exception as e:
        brier_df = pd.DataFrame({
            "horizon_week": horizons,
            "metric_name": ["brier_at_horizon"] * len(horizons),
            "metric_value": [np.nan] * len(horizons),
            "notes": [f"failed: {str(e)}"] * len(horizons),
        })

    support_rows = []
    calibration_rows = []
    pred_vs_obs_rows = []
    risk_auc_rows = []

    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
        eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
        eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

        support_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
            "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "metric_name": "risk_auc_at_horizon",
            "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
            "notes": "roc_auc_on_evaluable_subset",
        })

        pred_vs_obs_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df.shape[0] > 0:
            ranked = eval_df["pred_risk_h"].rank(method="first")
            n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))
            eval_df["calibration_bin"] = pd.qcut(
                ranked,
                q=n_bins_eff,
                labels=False,
                duplicates="drop"
            )

            calib_tab = (
                eval_df.groupby("calibration_bin")
                .agg(
                    n=("enrollment_id", "count"),
                    mean_predicted_risk=("pred_risk_h", "mean"),
                    observed_event_rate=("observed_event_by_h", "mean"),
                )
                .reset_index()
            )
            calib_tab["horizon_week"] = h
            calib_tab["abs_calibration_gap"] = (
                calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
            ).abs()

            calibration_rows.append({
                "horizon_week": h,
                "metric_name": "calibration_at_horizon",
                "metric_value": float(np.average(calib_tab["abs_calibration_gap"], weights=calib_tab["n"])),
                "notes": "Weighted absolute calibration gap across bins",
            })

    return {
        "primary": primary_df,
        "brier": pd.DataFrame(brier_df),
        "calibration": pd.DataFrame(calibration_rows),
        "secondary": pd.DataFrame(risk_auc_rows),
        "support": pd.DataFrame(support_rows),
        "pred_vs_obs": pd.DataFrame(pred_vs_obs_rows),
    }


In [ ]:

# ------------------------------
# 6) Train/evaluate ablation variants
# ------------------------------
results_primary = []
results_brier = []
results_calibration = []
results_secondary = []
results_support = []
results_pred_vs_obs = []
results_training_audit = []

for _, reg_row in ablation_registry_dt.iterrows():
    model_id = reg_row["model_id"]
    scenario_id = reg_row["scenario_id"]
    scenario_label = reg_row["scenario_label"]

    train_df = load_variant(reg_row["train_csv_path"], reg_row["train_parquet_path"])
    test_df = load_variant(reg_row["test_csv_path"], reg_row["test_parquet_path"])

    if "week" not in train_df.columns or "week" not in test_df.columns:
        raise ValueError(
            f"Scenario {model_id} / {scenario_id} does not contain 'week'. "
            "Discrete-time evaluation requires the structural time index."
        )

    feature_cols = get_feature_columns(train_df)
    preprocessor, numeric_cols, categorical_cols = build_preprocessor(feature_cols)

    X_train = preprocessor.fit_transform(train_df[feature_cols])
    X_test = preprocessor.transform(test_df[feature_cols])

    y_train = train_df[TARGET_COL].to_numpy()
    y_test = test_df[TARGET_COL].to_numpy()

    if hasattr(X_train, "toarray"):
        X_train_dense = X_train.toarray()
        X_test_dense = X_test.toarray()
    else:
        X_train_dense = X_train
        X_test_dense = X_test

    # ---------- model fit ----------
    if model_id == "linear_tuned":
        model = LogisticRegression(
            penalty="l1",
            C=0.1,
            solver="liblinear",
            class_weight=None,
            max_iter=2000,
            random_state=RANDOM_SEED,
        )
        model.fit(X_train_dense, y_train)
        pred_hazard = np.clip(model.predict_proba(X_test_dense)[:, 1], 1e-8, 1 - 1e-8)

    elif model_id == "neural_tuned":
        torch.manual_seed(RANDOM_SEED)
        np.random.seed(RANDOM_SEED)

        X_train_t = X_train_dense.astype(np.float32)
        X_test_t = X_test_dense.astype(np.float32)
        y_train_t = y_train.astype(np.float32)

        net = tt.practical.MLPVanilla(
            in_features=X_train_t.shape[1],
            num_nodes=[128, 64],
            out_features=1,
            batch_norm=True,
            dropout=0.1,
            output_bias=False,
        )
        model = tt.Model(net, torch.nn.BCEWithLogitsLoss(), tt.optim.AdamW)
        model.optimizer.set_lr(5e-4)

        _ = model.fit(
            X_train_t,
            y_train_t.reshape(-1, 1),
            batch_size=256,
            epochs=25,
            verbose=False,
        )

        logits = model.predict(X_test_t).reshape(-1)
        pred_hazard = 1.0 / (1.0 + np.exp(-logits))
        pred_hazard = np.clip(pred_hazard, 1e-8, 1 - 1e-8)

    else:
        raise ValueError(f"Unsupported model_id in P29: {model_id}")

    test_pred_df = test_df.copy()
    test_pred_df["pred_hazard"] = pred_hazard

    eval_outputs = evaluate_discrete_survival(test_pred_df, HORIZONS_WEEKS)

    primary_df = eval_outputs["primary"].copy()
    primary_df["model_id"] = model_id
    primary_df["scenario_id"] = scenario_id
    primary_df["scenario_label"] = scenario_label
    results_primary.append(primary_df)

    brier_df = eval_outputs["brier"].copy()
    brier_df["model_id"] = model_id
    brier_df["scenario_id"] = scenario_id
    brier_df["scenario_label"] = scenario_label
    results_brier.append(brier_df)

    calibration_df = eval_outputs["calibration"].copy()
    calibration_df["model_id"] = model_id
    calibration_df["scenario_id"] = scenario_id
    calibration_df["scenario_label"] = scenario_label
    results_calibration.append(calibration_df)

    secondary_df = eval_outputs["secondary"].copy()
    secondary_df["model_id"] = model_id
    secondary_df["scenario_id"] = scenario_id
    secondary_df["scenario_label"] = scenario_label
    results_secondary.append(secondary_df)

    support_df = eval_outputs["support"].copy()
    support_df["model_id"] = model_id
    support_df["scenario_id"] = scenario_id
    support_df["scenario_label"] = scenario_label
    results_support.append(support_df)

    pred_vs_obs_df = eval_outputs["pred_vs_obs"].copy()
    pred_vs_obs_df["model_id"] = model_id
    pred_vs_obs_df["scenario_id"] = scenario_id
    pred_vs_obs_df["scenario_label"] = scenario_label
    results_pred_vs_obs.append(pred_vs_obs_df)

    results_training_audit.append({
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "n_train_rows": int(train_df.shape[0]),
        "n_test_rows": int(test_df.shape[0]),
        "n_feature_columns_raw": int(len(feature_cols)),
        "n_numeric_features_raw": int(len(numeric_cols)),
        "n_categorical_features_raw": int(len(categorical_cols)),
        "n_features_after_transform": int(X_train_dense.shape[1]),
        "week_retained": bool("week" in feature_cols),
    })

# ------------------------------
# 7) Consolidate outputs
# ------------------------------
ablation_primary_df = pd.concat(results_primary, ignore_index=True)
ablation_brier_df = pd.concat(results_brier, ignore_index=True)
ablation_calibration_df = pd.concat(results_calibration, ignore_index=True)
ablation_secondary_df = pd.concat(results_secondary, ignore_index=True)
ablation_support_df = pd.concat(results_support, ignore_index=True)
ablation_pred_vs_obs_df = pd.concat(results_pred_vs_obs, ignore_index=True)
ablation_training_audit_df = pd.DataFrame(results_training_audit)

ablation_leaderboard_rows = []
for (model_id, scenario_id, scenario_label), g in ablation_primary_df.groupby(["model_id", "scenario_id", "scenario_label"]):
    row = {
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "ibs": float(g.loc[g["metric_name"] == "ibs", "metric_value"].iloc[0]),
        "c_index": float(g.loc[g["metric_name"] == "c_index", "metric_value"].iloc[0]),
    }
    for h in HORIZONS_WEEKS:
        row[f"brier_h{h}"] = float(
            ablation_brier_df[
                (ablation_brier_df["model_id"] == model_id) &
                (ablation_brier_df["scenario_id"] == scenario_id) &
                (ablation_brier_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"calibration_h{h}"] = float(
            ablation_calibration_df[
                (ablation_calibration_df["model_id"] == model_id) &
                (ablation_calibration_df["scenario_id"] == scenario_id) &
                (ablation_calibration_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"risk_auc_h{h}"] = float(
            ablation_secondary_df[
                (ablation_secondary_df["model_id"] == model_id) &
                (ablation_secondary_df["scenario_id"] == scenario_id) &
                (ablation_secondary_df["horizon_week"] == h) &
                (ablation_secondary_df["metric_name"] == "risk_auc_at_horizon")
            ]["metric_value"].iloc[0]
        )
    ablation_leaderboard_rows.append(row)

ablation_leaderboard_df = pd.DataFrame(ablation_leaderboard_rows).sort_values(
    by=["model_id", "ibs", "c_index"],
    ascending=[True, True, False]
).reset_index(drop=True)

delta_rows = []
for model_id in sorted(ablation_leaderboard_df["model_id"].unique()):
    ref = ablation_leaderboard_df[
        (ablation_leaderboard_df["model_id"] == model_id) &
        (ablation_leaderboard_df["scenario_id"] == "full_features")
    ].iloc[0]

    sub_df = ablation_leaderboard_df[ablation_leaderboard_df["model_id"] == model_id].copy()
    for _, r in sub_df.iterrows():
        delta = {
            "model_id": model_id,
            "scenario_id": r["scenario_id"],
            "scenario_label": r["scenario_label"],
            "delta_ibs_vs_full": float(r["ibs"] - ref["ibs"]),
            "delta_c_index_vs_full": float(r["c_index"] - ref["c_index"]),
        }
        for h in HORIZONS_WEEKS:
            delta[f"delta_brier_h{h}_vs_full"] = float(r[f"brier_h{h}"] - ref[f"brier_h{h}"])
            delta[f"delta_calibration_h{h}_vs_full"] = float(r[f"calibration_h{h}"] - ref[f"calibration_h{h}"])
            delta[f"delta_risk_auc_h{h}_vs_full"] = float(r[f"risk_auc_h{h}"] - ref[f"risk_auc_h{h}"])
        delta_rows.append(delta)

ablation_delta_vs_full_df = pd.DataFrame(delta_rows).sort_values(
    by=["model_id", "scenario_id"]
).reset_index(drop=True)

# ------------------------------
# 8) Save artifacts
# ------------------------------
primary_path = TABLES_DIR / "table_ablation_discrete_primary_metrics.csv"
brier_path = TABLES_DIR / "table_ablation_discrete_brier_by_horizon.csv"
calibration_path = TABLES_DIR / "table_ablation_discrete_calibration_by_horizon.csv"
secondary_path = TABLES_DIR / "table_ablation_discrete_secondary_metrics.csv"
support_path = TABLES_DIR / "table_ablation_discrete_support_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_ablation_discrete_predicted_vs_observed_survival.csv"
audit_path = TABLES_DIR / "table_ablation_discrete_training_audit.csv"
leaderboard_path = TABLES_DIR / "table_ablation_discrete_leaderboard.csv"
delta_path = TABLES_DIR / "table_ablation_discrete_delta_vs_full.csv"
config_path = METADATA_DIR / "ablation_discrete_run_summary.json"

ablation_primary_df.to_csv(primary_path, index=False)
ablation_brier_df.to_csv(brier_path, index=False)
ablation_calibration_df.to_csv(calibration_path, index=False)
ablation_secondary_df.to_csv(secondary_path, index=False)
ablation_support_df.to_csv(support_path, index=False)
ablation_pred_vs_obs_df.to_csv(pred_vs_obs_path, index=False)
ablation_training_audit_df.to_csv(audit_path, index=False)
ablation_leaderboard_df.to_csv(leaderboard_path, index=False)
ablation_delta_vs_full_df.to_csv(delta_path, index=False)

save_json(
    {
        "models_run": sorted(ablation_leaderboard_df["model_id"].unique().tolist()),
        "scenarios_run": sorted(ablation_leaderboard_df["scenario_id"].unique().tolist()),
        "horizons": [int(h) for h in HORIZONS_WEEKS],
        "n_total_runs": int(ablation_leaderboard_df.shape[0]),
        "week_always_retained_for_discrete_time": True,
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation discrete training audit:")
display(ablation_training_audit_df)

print("\nAblation discrete leaderboard:")
display(ablation_leaderboard_df)

print("\nAblation discrete delta vs full:")
display(ablation_delta_vs_full_df)

print("\nSaved:")
print("-", primary_path.resolve())
print("-", brier_path.resolve())
print("-", calibration_path.resolve())
print("-", secondary_path.resolve())
print("-", support_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", audit_path.resolve())
print("-", leaderboard_path.resolve())
print("-", delta_path.resolve())
print("-", config_path.resolve())


# F4 — Train and Evaluate Ablation Variants for Cox and DeepSurv Tuned Models

In [ ]:
# ==============================================================
# F4 — Train and Evaluate Ablation Variants for Cox and DeepSurv Tuned Models
# --------------------------------------------------------------
# Purpose:
#   Train and evaluate the ablation variants for the tuned
#   continuous-time model families:
#     - cox_tuned
#     - deepsurv_tuned
#
# Methodological note:
#   This step reuses the benchmark split and the benchmark
#   evaluation protocol. The objective is to measure how much
#   performance changes when static or early-window feature
#   blocks are removed or isolated.
# ==============================================================

print("\n" + "=" * 70)
print("F4 — Train and Evaluate Ablation Variants for Cox and DeepSurv Tuned Models")
print("=" * 70)
print("Methodological note: this step trains and evaluates ablation")
print("variants for the tuned continuous-time families only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED", "save_json"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from lifelines import CoxPHFitter
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

try:
    from pycox.evaluation import EvalSurv
    from pycox.models import CoxPH
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P30.")

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P30.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
except Exception:
    pass

# ------------------------------
# 3) Registry / paths
# ------------------------------
DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "models"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_REGISTRY_PATH = TABLES_DIR / "table_ablation_variant_registry.csv"

if not ABLATION_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing ablation registry: {ABLATION_REGISTRY_PATH}")

ablation_registry = pd.read_csv(ABLATION_REGISTRY_PATH)

TARGET_MODELS = ["cox_tuned", "deepsurv_tuned"]

ablation_registry_ct = ablation_registry[
    ablation_registry["model_id"].isin(TARGET_MODELS)
].copy()

# ------------------------------
# 4) Column definitions
# ------------------------------
AUX_ENROLLMENT = [
    "enrollment_id",
    "id_student",
    "code_module",
    "code_presentation",
    "duration",
    "duration_raw",
    "used_zero_week_fallback_for_censoring",
    "split",
    "time_for_split",
    "time_bucket",
    "event_time_bucket_label",
]

TARGET_COL = "event"
DURATION_COL = "duration"

CATEGORICAL_FEATURES = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]


### Funcao: load_variant

Definicao isolada de load_variant para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 5) Helpers
# ------------------------------
def load_variant(path_csv: str, path_parquet: str) -> pd.DataFrame:
    p_parquet = Path(path_parquet)
    p_csv = Path(path_csv)
    if p_parquet.exists():
        return pd.read_parquet(p_parquet)
    if p_csv.exists():
        return pd.read_csv(p_csv)
    raise FileNotFoundError(f"Variant file not found:\n- {p_parquet}\n- {p_csv}")


### Funcao: get_feature_columns

Definicao isolada de get_feature_columns para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_columns(df: pd.DataFrame):
    excluded = set(AUX_ENROLLMENT + [TARGET_COL])
    return [c for c in df.columns if c not in excluded]


### Funcao: split_feature_types

Definicao isolada de split_feature_types para reutilizacao nas etapas seguintes.


In [ ]:

def split_feature_types(feature_cols):
    categorical_cols = [c for c in feature_cols if c in CATEGORICAL_FEATURES]
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]
    return numeric_cols, categorical_cols


### Funcao: build_preprocessor

Definicao isolada de build_preprocessor para reutilizacao nas etapas seguintes.


In [ ]:

def build_preprocessor(feature_cols):
    numeric_cols, categorical_cols = split_feature_types(feature_cols)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor, numeric_cols, categorical_cols


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]


### Funcao: evaluate_continuous_survival

Definicao isolada de evaluate_continuous_survival para reutilizacao nas etapas seguintes.


In [ ]:

def evaluate_continuous_survival(surv_df: pd.DataFrame, truth_test: pd.DataFrame, horizons: list[int]):
    durations_test = truth_test["duration"].astype(int).to_numpy()
    events_test = truth_test["event"].astype(int).to_numpy()

    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    primary_rows = []
    try:
        c_index = float(eval_surv.concordance_td())
        c_index_note = "pycox_concordance_td"
    except Exception as e:
        c_index = np.nan
        c_index_note = f"failed: {str(e)}"

    try:
        max_requested_horizon = int(max(horizons))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
        ibs_note = "pycox_integrated_brier_score"
    except Exception as e:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)}"

    primary_rows.append({"metric_name": "ibs", "metric_value": ibs_value, "notes": ibs_note})
    primary_rows.append({"metric_name": "c_index", "metric_value": c_index, "notes": c_index_note})
    primary_df = pd.DataFrame(primary_rows)

    try:
        brier_h = eval_surv.brier_score(np.array(horizons, dtype=int))
        brier_df = pd.DataFrame({
            "horizon_week": list(brier_h.index.astype(int)),
            "metric_name": ["brier_at_horizon"] * len(brier_h.index),
            "metric_value": list(brier_h.values.astype(float)),
            "notes": ["pycox_brier_score"] * len(brier_h.index),
        })
    except Exception as e:
        brier_df = pd.DataFrame({
            "horizon_week": horizons,
            "metric_name": ["brier_at_horizon"] * len(horizons),
            "metric_value": [np.nan] * len(horizons),
            "notes": [f"failed: {str(e)}"] * len(horizons),
        })

    support_rows = []
    calibration_rows = []
    pred_vs_obs_rows = []
    risk_auc_rows = []

    for h in horizons:
        pred_surv_h = get_surv_at_horizon(surv_df, h)
        pred_risk_h = 1.0 - pred_surv_h

        eval_df = truth_test.copy()
        eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
        eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

        eval_df["is_evaluable_at_h"] = (
            ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
            (eval_df["duration"] >= h)
        ).astype(int)

        eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
        eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
        eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

        support_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
            "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df["observed_event_by_h"].nunique() >= 2:
            risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
        else:
            risk_auc = np.nan

        risk_auc_rows.append({
            "horizon_week": h,
            "metric_name": "risk_auc_at_horizon",
            "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
            "notes": "roc_auc_on_evaluable_subset",
        })

        pred_vs_obs_rows.append({
            "horizon_week": h,
            "n_evaluable_enrollments": int(eval_df.shape[0]),
            "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
            "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
        })

        if eval_df.shape[0] > 0:
            ranked = eval_df["pred_risk_h"].rank(method="first")
            n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))
            eval_df["calibration_bin"] = pd.qcut(
                ranked,
                q=n_bins_eff,
                labels=False,
                duplicates="drop"
            )

            calib_tab = (
                eval_df.groupby("calibration_bin")
                .agg(
                    n=("enrollment_id", "count"),
                    mean_predicted_risk=("pred_risk_h", "mean"),
                    observed_event_rate=("observed_event_by_h", "mean"),
                )
                .reset_index()
            )
            calib_tab["horizon_week"] = h
            calib_tab["abs_calibration_gap"] = (
                calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
            ).abs()

            calibration_rows.append({
                "horizon_week": h,
                "metric_name": "calibration_at_horizon",
                "metric_value": float(np.average(calib_tab["abs_calibration_gap"], weights=calib_tab["n"])),
                "notes": "Weighted absolute calibration gap across bins",
            })

    return {
        "primary": primary_df,
        "brier": pd.DataFrame(brier_df),
        "calibration": pd.DataFrame(calibration_rows),
        "secondary": pd.DataFrame(risk_auc_rows),
        "support": pd.DataFrame(support_rows),
        "pred_vs_obs": pd.DataFrame(pred_vs_obs_rows),
    }


In [ ]:

# ------------------------------
# 6) Train/evaluate ablation variants
# ------------------------------
results_primary = []
results_brier = []
results_calibration = []
results_secondary = []
results_support = []
results_pred_vs_obs = []
results_training_audit = []

for _, reg_row in ablation_registry_ct.iterrows():
    model_id = reg_row["model_id"]
    scenario_id = reg_row["scenario_id"]
    scenario_label = reg_row["scenario_label"]

    train_df = load_variant(reg_row["train_csv_path"], reg_row["train_parquet_path"])
    test_df = load_variant(reg_row["test_csv_path"], reg_row["test_parquet_path"])

    feature_cols = get_feature_columns(train_df)
    preprocessor, numeric_cols, categorical_cols = build_preprocessor(feature_cols)

    X_train = preprocessor.fit_transform(train_df[feature_cols])
    X_test = preprocessor.transform(test_df[feature_cols])

    if hasattr(X_train, "toarray"):
        X_train_dense = X_train.toarray().astype(np.float32)
        X_test_dense = X_test.toarray().astype(np.float32)
    else:
        X_train_dense = np.asarray(X_train).astype(np.float32)
        X_test_dense = np.asarray(X_test).astype(np.float32)

    y_train_event = train_df[TARGET_COL].to_numpy().astype(np.int32)
    y_test_event = test_df[TARGET_COL].to_numpy().astype(np.int32)
    y_train_duration = train_df[DURATION_COL].to_numpy().astype(np.float32)
    y_test_duration = test_df[DURATION_COL].to_numpy().astype(np.float32)

    # ---------- model fit ----------
    if model_id == "cox_tuned":
        feature_names = [f"x{i}" for i in range(X_train_dense.shape[1])]
        train_fit_df = pd.DataFrame(X_train_dense, columns=feature_names)
        train_fit_df["duration"] = y_train_duration
        train_fit_df["event"] = y_train_event

        # train-only zero-variance filter
        stds = train_fit_df[feature_names].std(axis=0, ddof=0)
        keep_feature_names = stds[stds > 0].index.tolist()

        train_fit_df = train_fit_df[keep_feature_names + ["duration", "event"]].copy()
        test_fit_df = pd.DataFrame(X_test_dense, columns=feature_names)[keep_feature_names].copy()

        model = CoxPHFitter(
            penalizer=0.001,
            l1_ratio=0.0,
        )
        model.fit(
            train_fit_df,
            duration_col="duration",
            event_col="event",
            show_progress=False,
        )

        surv_df = model.predict_survival_function(
            test_fit_df,
            times=np.arange(0, int(np.max(y_test_duration)) + 1, dtype=int)
        ).copy()

    elif model_id == "deepsurv_tuned":
        torch.manual_seed(RANDOM_SEED)
        np.random.seed(RANDOM_SEED)

        net = tt.practical.MLPVanilla(
            in_features=X_train_dense.shape[1],
            num_nodes=[64, 32],
            out_features=1,
            batch_norm=True,
            dropout=0.3,
            output_bias=False,
        )

        model = CoxPH(net, tt.optim.Adam)
        model.optimizer.set_lr(5e-4)

        _ = model.fit(
            X_train_dense,
            (y_train_duration, y_train_event),
            batch_size=256,
            epochs=55,
            verbose=False,
        )

        _ = model.compute_baseline_hazards()
        surv_df = model.predict_surv_df(X_test_dense)

    else:
        raise ValueError(f"Unsupported model_id in P30: {model_id}")

    surv_df.columns = test_df["enrollment_id"].tolist()
    surv_df.columns.name = "enrollment_id"
    surv_df.index = surv_df.index.astype(int)

    truth_test = test_df[["enrollment_id", "event", "duration"]].copy()

    eval_outputs = evaluate_continuous_survival(surv_df, truth_test, HORIZONS_WEEKS)

    primary_df = eval_outputs["primary"].copy()
    primary_df["model_id"] = model_id
    primary_df["scenario_id"] = scenario_id
    primary_df["scenario_label"] = scenario_label
    results_primary.append(primary_df)

    brier_df = eval_outputs["brier"].copy()
    brier_df["model_id"] = model_id
    brier_df["scenario_id"] = scenario_id
    brier_df["scenario_label"] = scenario_label
    results_brier.append(brier_df)

    calibration_df = eval_outputs["calibration"].copy()
    calibration_df["model_id"] = model_id
    calibration_df["scenario_id"] = scenario_id
    calibration_df["scenario_label"] = scenario_label
    results_calibration.append(calibration_df)

    secondary_df = eval_outputs["secondary"].copy()
    secondary_df["model_id"] = model_id
    secondary_df["scenario_id"] = scenario_id
    secondary_df["scenario_label"] = scenario_label
    results_secondary.append(secondary_df)

    support_df = eval_outputs["support"].copy()
    support_df["model_id"] = model_id
    support_df["scenario_id"] = scenario_id
    support_df["scenario_label"] = scenario_label
    results_support.append(support_df)

    pred_vs_obs_df = eval_outputs["pred_vs_obs"].copy()
    pred_vs_obs_df["model_id"] = model_id
    pred_vs_obs_df["scenario_id"] = scenario_id
    pred_vs_obs_df["scenario_label"] = scenario_label
    results_pred_vs_obs.append(pred_vs_obs_df)

    results_training_audit.append({
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "n_train_rows": int(train_df.shape[0]),
        "n_test_rows": int(test_df.shape[0]),
        "n_feature_columns_raw": int(len(feature_cols)),
        "n_numeric_features_raw": int(len(numeric_cols)),
        "n_categorical_features_raw": int(len(categorical_cols)),
        "n_features_after_transform": int(X_train_dense.shape[1]),
    })

# ------------------------------
# 7) Consolidate outputs
# ------------------------------
ablation_primary_df = pd.concat(results_primary, ignore_index=True)
ablation_brier_df = pd.concat(results_brier, ignore_index=True)
ablation_calibration_df = pd.concat(results_calibration, ignore_index=True)
ablation_secondary_df = pd.concat(results_secondary, ignore_index=True)
ablation_support_df = pd.concat(results_support, ignore_index=True)
ablation_pred_vs_obs_df = pd.concat(results_pred_vs_obs, ignore_index=True)
ablation_training_audit_df = pd.DataFrame(results_training_audit)

ablation_leaderboard_rows = []
for (model_id, scenario_id, scenario_label), g in ablation_primary_df.groupby(["model_id", "scenario_id", "scenario_label"]):
    row = {
        "model_id": model_id,
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "ibs": float(g.loc[g["metric_name"] == "ibs", "metric_value"].iloc[0]),
        "c_index": float(g.loc[g["metric_name"] == "c_index", "metric_value"].iloc[0]),
    }
    for h in HORIZONS_WEEKS:
        row[f"brier_h{h}"] = float(
            ablation_brier_df[
                (ablation_brier_df["model_id"] == model_id) &
                (ablation_brier_df["scenario_id"] == scenario_id) &
                (ablation_brier_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"calibration_h{h}"] = float(
            ablation_calibration_df[
                (ablation_calibration_df["model_id"] == model_id) &
                (ablation_calibration_df["scenario_id"] == scenario_id) &
                (ablation_calibration_df["horizon_week"] == h)
            ]["metric_value"].iloc[0]
        )
        row[f"risk_auc_h{h}"] = float(
            ablation_secondary_df[
                (ablation_secondary_df["model_id"] == model_id) &
                (ablation_secondary_df["scenario_id"] == scenario_id) &
                (ablation_secondary_df["horizon_week"] == h) &
                (ablation_secondary_df["metric_name"] == "risk_auc_at_horizon")
            ]["metric_value"].iloc[0]
        )
    ablation_leaderboard_rows.append(row)

ablation_leaderboard_df = pd.DataFrame(ablation_leaderboard_rows).sort_values(
    by=["model_id", "ibs", "c_index"],
    ascending=[True, True, False]
).reset_index(drop=True)

delta_rows = []
for model_id in sorted(ablation_leaderboard_df["model_id"].unique()):
    ref = ablation_leaderboard_df[
        (ablation_leaderboard_df["model_id"] == model_id) &
        (ablation_leaderboard_df["scenario_id"] == "full_features")
    ].iloc[0]

    sub_df = ablation_leaderboard_df[ablation_leaderboard_df["model_id"] == model_id].copy()
    for _, r in sub_df.iterrows():
        delta = {
            "model_id": model_id,
            "scenario_id": r["scenario_id"],
            "scenario_label": r["scenario_label"],
            "delta_ibs_vs_full": float(r["ibs"] - ref["ibs"]),
            "delta_c_index_vs_full": float(r["c_index"] - ref["c_index"]),
        }
        for h in HORIZONS_WEEKS:
            delta[f"delta_brier_h{h}_vs_full"] = float(r[f"brier_h{h}"] - ref[f"brier_h{h}"])
            delta[f"delta_calibration_h{h}_vs_full"] = float(r[f"calibration_h{h}"] - ref[f"calibration_h{h}"])
            delta[f"delta_risk_auc_h{h}_vs_full"] = float(r[f"risk_auc_h{h}"] - ref[f"risk_auc_h{h}"])
        delta_rows.append(delta)

ablation_delta_vs_full_df = pd.DataFrame(delta_rows).sort_values(
    by=["model_id", "scenario_id"]
).reset_index(drop=True)

# ------------------------------
# 8) Save artifacts
# ------------------------------
primary_path = TABLES_DIR / "table_ablation_continuous_primary_metrics.csv"
brier_path = TABLES_DIR / "table_ablation_continuous_brier_by_horizon.csv"
calibration_path = TABLES_DIR / "table_ablation_continuous_calibration_by_horizon.csv"
secondary_path = TABLES_DIR / "table_ablation_continuous_secondary_metrics.csv"
support_path = TABLES_DIR / "table_ablation_continuous_support_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_ablation_continuous_predicted_vs_observed_survival.csv"
audit_path = TABLES_DIR / "table_ablation_continuous_training_audit.csv"
leaderboard_path = TABLES_DIR / "table_ablation_continuous_leaderboard.csv"
delta_path = TABLES_DIR / "table_ablation_continuous_delta_vs_full.csv"
config_path = METADATA_DIR / "ablation_continuous_run_summary.json"

ablation_primary_df.to_csv(primary_path, index=False)
ablation_brier_df.to_csv(brier_path, index=False)
ablation_calibration_df.to_csv(calibration_path, index=False)
ablation_secondary_df.to_csv(secondary_path, index=False)
ablation_support_df.to_csv(support_path, index=False)
ablation_pred_vs_obs_df.to_csv(pred_vs_obs_path, index=False)
ablation_training_audit_df.to_csv(audit_path, index=False)
ablation_leaderboard_df.to_csv(leaderboard_path, index=False)
ablation_delta_vs_full_df.to_csv(delta_path, index=False)

save_json(
    {
        "models_run": sorted(ablation_leaderboard_df["model_id"].unique().tolist()),
        "scenarios_run": sorted(ablation_leaderboard_df["scenario_id"].unique().tolist()),
        "horizons": [int(h) for h in HORIZONS_WEEKS],
        "n_total_runs": int(ablation_leaderboard_df.shape[0]),
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation continuous training audit:")
display(ablation_training_audit_df)

print("\nAblation continuous leaderboard:")
display(ablation_leaderboard_df)

print("\nAblation continuous delta vs full:")
display(ablation_delta_vs_full_df)

print("\nSaved:")
print("-", primary_path.resolve())
print("-", brier_path.resolve())
print("-", calibration_path.resolve())
print("-", secondary_path.resolve())
print("-", support_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", audit_path.resolve())
print("-", leaderboard_path.resolve())
print("-", delta_path.resolve())
print("-", config_path.resolve())


# F5 — Consolidate Ablation Results Across All Tuned Families

In [ ]:
# ==============================================================
# F5 — Consolidate Ablation Results Across All Tuned Families
# --------------------------------------------------------------
# Purpose:
#   Consolidate the ablation results from:
#     - discrete-time tuned families (F3)
#     - continuous-time tuned families (F4)
#
# Methodological note:
#   This step does not train any model. It only consolidates the
#   ablation outputs already generated in previous steps.
#
# Main goals:
#   - create a unified ablation leaderboard
#   - compare deltas vs full_features across families
#   - prepare paper-friendly summary tables
# ==============================================================

print("\n" + "=" * 70)
print("F5 — Consolidate Ablation Results Across All Tuned Families")
print("=" * 70)
print("Methodological note: this step consolidates ablation outputs only.")
print("No model is trained here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json", "HORIZONS_WEEKS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------
# 2) Required files
# ------------------------------
DISCRETE_LEADERBOARD_PATH = TABLES_DIR / "table_ablation_discrete_leaderboard.csv"
DISCRETE_DELTA_PATH = TABLES_DIR / "table_ablation_discrete_delta_vs_full.csv"

CONTINUOUS_LEADERBOARD_PATH = TABLES_DIR / "table_ablation_continuous_leaderboard.csv"
CONTINUOUS_DELTA_PATH = TABLES_DIR / "table_ablation_continuous_delta_vs_full.csv"

required_paths = [
    DISCRETE_LEADERBOARD_PATH,
    DISCRETE_DELTA_PATH,
    CONTINUOUS_LEADERBOARD_PATH,
    CONTINUOUS_DELTA_PATH,
]

missing_files = [str(p) for p in required_paths if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required ablation result files:\n- " + "\n- ".join(missing_files)
    )

# ------------------------------
# 3) Load inputs
# ------------------------------
ablation_discrete_leaderboard = pd.read_csv(DISCRETE_LEADERBOARD_PATH)
ablation_discrete_delta = pd.read_csv(DISCRETE_DELTA_PATH)

ablation_continuous_leaderboard = pd.read_csv(CONTINUOUS_LEADERBOARD_PATH)
ablation_continuous_delta = pd.read_csv(CONTINUOUS_DELTA_PATH)

# ------------------------------
# 4) Consolidate leaderboards
# ------------------------------
ablation_leaderboard_all = pd.concat(
    [ablation_discrete_leaderboard, ablation_continuous_leaderboard],
    ignore_index=True
)

family_map = {
    "linear_tuned": "discrete_time_linear",
    "neural_tuned": "discrete_time_neural",
    "cox_tuned": "continuous_time_cox",
    "deepsurv_tuned": "continuous_time_deepsurv",
}

display_name_map = {
    "linear_tuned": "Linear Discrete-Time Hazard (Tuned)",
    "neural_tuned": "Neural Discrete-Time Survival (Tuned)",
    "cox_tuned": "Cox Comparable (Tuned)",
    "deepsurv_tuned": "DeepSurv (Tuned)",
}

ablation_leaderboard_all["family"] = ablation_leaderboard_all["model_id"].map(family_map)
ablation_leaderboard_all["display_name"] = ablation_leaderboard_all["model_id"].map(display_name_map)

ablation_leaderboard_all = ablation_leaderboard_all[
    [
        "model_id", "display_name", "family", "scenario_id", "scenario_label",
        "ibs", "c_index",
        "brier_h10", "calibration_h10", "risk_auc_h10",
        "brier_h20", "calibration_h20", "risk_auc_h20",
        "brier_h30", "calibration_h30", "risk_auc_h30",
    ]
].copy()

ablation_leaderboard_all = ablation_leaderboard_all.sort_values(
    by=["model_id", "ibs", "c_index"],
    ascending=[True, True, False]
).reset_index(drop=True)

# ------------------------------
# 5) Consolidate deltas
# ------------------------------
ablation_delta_all = pd.concat(
    [ablation_discrete_delta, ablation_continuous_delta],
    ignore_index=True
)

ablation_delta_all["family"] = ablation_delta_all["model_id"].map(family_map)
ablation_delta_all["display_name"] = ablation_delta_all["model_id"].map(display_name_map)

ablation_delta_all = ablation_delta_all[
    [
        "model_id", "display_name", "family", "scenario_id", "scenario_label",
        "delta_ibs_vs_full", "delta_c_index_vs_full",
        "delta_brier_h10_vs_full", "delta_calibration_h10_vs_full", "delta_risk_auc_h10_vs_full",
        "delta_brier_h20_vs_full", "delta_calibration_h20_vs_full", "delta_risk_auc_h20_vs_full",
        "delta_brier_h30_vs_full", "delta_calibration_h30_vs_full", "delta_risk_auc_h30_vs_full",
    ]
].copy()

ablation_delta_all = ablation_delta_all.sort_values(
    by=["model_id", "scenario_id"]
).reset_index(drop=True)

# ------------------------------
# 6) Paper-friendly summary:
#    compare scenario effects per model
# ------------------------------
summary_rows = []

for model_id in sorted(ablation_delta_all["model_id"].unique()):
    sub = ablation_delta_all[ablation_delta_all["model_id"] == model_id].copy()

    def row_for(scenario):
        r = sub[sub["scenario_id"] == scenario]
        return r.iloc[0] if not r.empty else None

    full_r = row_for("full_features")
    drop_static_r = row_for("drop_static_structural")
    drop_temporal_r = row_for("drop_temporal_signal")

    summary_rows.append({
        "model_id": model_id,
        "display_name": display_name_map.get(model_id, model_id),
        "family": family_map.get(model_id, "unknown"),
        "delta_ibs_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_ibs_vs_full"]),
        "delta_ibs_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_ibs_vs_full"]),
        "delta_c_index_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_c_index_vs_full"]),
        "delta_c_index_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_c_index_vs_full"]),
        "delta_risk_auc_h10_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_risk_auc_h10_vs_full"]),
        "delta_risk_auc_h10_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_risk_auc_h10_vs_full"]),
        "delta_risk_auc_h20_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_risk_auc_h20_vs_full"]),
        "delta_risk_auc_h20_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_risk_auc_h20_vs_full"]),
        "delta_risk_auc_h30_drop_static": np.nan if drop_static_r is None else float(drop_static_r["delta_risk_auc_h30_vs_full"]),
        "delta_risk_auc_h30_drop_temporal": np.nan if drop_temporal_r is None else float(drop_temporal_r["delta_risk_auc_h30_vs_full"]),
    })

ablation_summary_by_model = pd.DataFrame(summary_rows)

# dominance ratio / contrast helper
ablation_summary_by_model["ibs_temporal_vs_static_ratio"] = (
    ablation_summary_by_model["delta_ibs_drop_temporal"] /
    ablation_summary_by_model["delta_ibs_drop_static"].replace(0, np.nan)
)

ablation_summary_by_model["abs_cindex_temporal_vs_static_ratio"] = (
    ablation_summary_by_model["delta_c_index_drop_temporal"].abs() /
    ablation_summary_by_model["delta_c_index_drop_static"].abs().replace(0, np.nan)
)

# ------------------------------
# 7) Cross-family scenario comparison
# ------------------------------
scenario_comparison_rows = []

SCENARIOS_KEEP = [
    "full_features",
    "drop_static_structural",
    "drop_temporal_signal",
]

for scenario_id in SCENARIOS_KEEP:
    sub = ablation_leaderboard_all[ablation_leaderboard_all["scenario_id"] == scenario_id].copy()
    sub = sub.sort_values(by=["ibs", "c_index"], ascending=[True, False]).reset_index(drop=True)
    sub["rank_ibs"] = sub["ibs"].rank(method="min", ascending=True)
    sub["rank_c_index"] = sub["c_index"].rank(method="min", ascending=False)
    scenario_comparison_rows.append(sub)

ablation_scenario_comparison = pd.concat(scenario_comparison_rows, ignore_index=True)

# ------------------------------
# 8) Save outputs
# ------------------------------
leaderboard_all_path = TABLES_DIR / "table_ablation_leaderboard_all_tuned.csv"
delta_all_path = TABLES_DIR / "table_ablation_delta_all_tuned.csv"
summary_by_model_path = TABLES_DIR / "table_ablation_summary_by_model.csv"
scenario_comparison_path = TABLES_DIR / "table_ablation_scenario_comparison.csv"
config_path = METADATA_DIR / "ablation_consolidated_summary.json"

ablation_leaderboard_all.to_csv(leaderboard_all_path, index=False)
ablation_delta_all.to_csv(delta_all_path, index=False)
ablation_summary_by_model.to_csv(summary_by_model_path, index=False)
ablation_scenario_comparison.to_csv(scenario_comparison_path, index=False)

save_json(
    {
        "n_models": int(ablation_leaderboard_all["model_id"].nunique()),
        "n_rows_leaderboard": int(ablation_leaderboard_all.shape[0]),
        "n_rows_delta": int(ablation_delta_all.shape[0]),
        "scenarios_included": sorted(ablation_leaderboard_all["scenario_id"].unique().tolist()),
        "horizons": [int(h) for h in HORIZONS_WEEKS],
    },
    config_path,
)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nAblation leaderboard — all tuned families:")
display(ablation_leaderboard_all)

print("\nAblation delta — all tuned families:")
display(ablation_delta_all)

print("\nAblation summary by model:")
display(ablation_summary_by_model)

print("\nAblation scenario comparison:")
display(ablation_scenario_comparison)

print("\nSaved:")
print("-", leaderboard_all_path.resolve())
print("-", delta_all_path.resolve())
print("-", summary_by_model_path.resolve())
print("-", scenario_comparison_path.resolve())
print("-", config_path.resolve())

# F6 — Consolidated Preprocessing and Tuning Audit

In [ ]:
# ==============================================================
# F6 — Consolidated Preprocessing and Tuning Audit
# --------------------------------------------------------------
# Purpose:
#   Consolidate preprocessing and tuning documentation across the
#   four tuned benchmark model families into a single audit table
#   suitable for editorial freezing and appendix use.
#
# Methodological note:
#   This step does not retrain any model.
#   It reuses previously exported preprocessing summaries,
#   preprocessing configs, tuning configs, and tuning-result tables.
#
# Scope:
#   Tuned benchmark families only:
#   - linear_tuned
#   - neural_tuned
#   - cox_tuned
#   - deepsurv_tuned
#
# Outputs:
#   - table_preprocessing_and_tuning_audit.csv
#   - preprocessing_and_tuning_audit_summary.json
# ==============================================================

print("\n" + "=" * 70)
print("F6 — Consolidated Preprocessing and Tuning Audit")
print("=" * 70)
print("Methodological note: this step consolidates preprocessing and")
print("tuning documentation across the tuned benchmark families only.")
print("No model is retrained and no benchmark metric is recomputed here.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import json
import numpy as np
import pandas as pd


### Funcao: read_json

Definicao isolada de read_json para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 2) Helper functions
# ------------------------------
def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


### Funcao: must_exist

Definicao isolada de must_exist para reutilizacao nas etapas seguintes.


In [ ]:

def must_exist(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing required {label}: {path}")
    return path


### Funcao: safe_first

Definicao isolada de safe_first para reutilizacao nas etapas seguintes.


In [ ]:

def safe_first(df: pd.DataFrame, col: str, default=np.nan):
    if col in df.columns and df.shape[0] > 0:
        return df.iloc[0][col]
    return default


### Funcao: stringify_search_space

Definicao isolada de stringify_search_space para reutilizacao nas etapas seguintes.


In [ ]:

def stringify_search_space(value):
    if value is None:
        return "not_reported"
    if isinstance(value, dict):
        parts = []
        for k, v in value.items():
            parts.append(f"{k}={v}")
        return "; ".join(parts)
    return str(value)


### Funcao: stringify_list_field

Definicao isolada de stringify_list_field para reutilizacao nas etapas seguintes.


In [ ]:

def stringify_list_field(value):
    if value is None:
        return "not_reported"
    if isinstance(value, list):
        return ", ".join(str(x) for x in value)
    return str(value)


In [ ]:

# ------------------------------
# 3) Resolve inputs
# ------------------------------
# preprocessing summaries / configs
linear_preproc_summary_path = must_exist(
    TABLES_DIR / "table_linear_preprocessing_summary.csv",
    "linear preprocessing summary"
)
neural_preproc_summary_path = must_exist(
    TABLES_DIR / "table_neural_preprocessing_summary.csv",
    "neural preprocessing summary"
)
cox_preproc_summary_path = must_exist(
    TABLES_DIR / "table_cox_preprocessing_summary.csv",
    "cox preprocessing summary"
)
deepsurv_preproc_summary_path = must_exist(
    TABLES_DIR / "table_deepsurv_preprocessing_summary.csv",
    "deepsurv preprocessing summary"
)

linear_preproc_config_path = must_exist(
    METADATA_DIR / "linear_preprocessing_config.json",
    "linear preprocessing config"
)
neural_preproc_config_path = must_exist(
    METADATA_DIR / "neural_preprocessing_config.json",
    "neural preprocessing config"
)
cox_preproc_config_path = must_exist(
    METADATA_DIR / "cox_preprocessing_config.json",
    "cox preprocessing config"
)
deepsurv_preproc_config_path = must_exist(
    METADATA_DIR / "deepsurv_preprocessing_config.json",
    "deepsurv preprocessing config"
)

# tuning results / configs
linear_tuning_results_path = must_exist(
    TABLES_DIR / "table_linear_tuning_results.csv",
    "linear tuning results"
)
neural_tuning_results_path = must_exist(
    TABLES_DIR / "table_neural_tuning_results.csv",
    "neural tuning results"
)
cox_tuning_results_path = must_exist(
    TABLES_DIR / "table_cox_tuning_results.csv",
    "cox tuning results"
)
deepsurv_tuning_results_path = must_exist(
    TABLES_DIR / "table_deepsurv_tuning_results.csv",
    "deepsurv tuning results"
)

linear_tuning_config_path = must_exist(
    METADATA_DIR / "linear_tuned_model_config.json",
    "linear tuned model config"
)
neural_tuning_config_path = must_exist(
    METADATA_DIR / "neural_tuned_model_config.json",
    "neural tuned model config"
)
cox_tuning_config_path = must_exist(
    METADATA_DIR / "cox_tuned_model_config.json",
    "cox tuned model config"
)
deepsurv_tuning_config_path = must_exist(
    METADATA_DIR / "deepsurv_tuned_model_config.json",
    "deepsurv tuned model config"
)

# ------------------------------
# 4) Load artifacts
# ------------------------------
linear_preproc_summary = pd.read_csv(linear_preproc_summary_path)
neural_preproc_summary = pd.read_csv(neural_preproc_summary_path)
cox_preproc_summary = pd.read_csv(cox_preproc_summary_path)
deepsurv_preproc_summary = pd.read_csv(deepsurv_preproc_summary_path)

linear_preproc_config = read_json(linear_preproc_config_path)
neural_preproc_config = read_json(neural_preproc_config_path)
cox_preproc_config = read_json(cox_preproc_config_path)
deepsurv_preproc_config = read_json(deepsurv_preproc_config_path)

linear_tuning_results = pd.read_csv(linear_tuning_results_path)
neural_tuning_results = pd.read_csv(neural_tuning_results_path)
cox_tuning_results = pd.read_csv(cox_tuning_results_path)
deepsurv_tuning_results = pd.read_csv(deepsurv_tuning_results_path)

linear_tuning_config = read_json(linear_tuning_config_path)
neural_tuning_config = read_json(neural_tuning_config_path)
cox_tuning_config = read_json(cox_tuning_config_path)
deepsurv_tuning_config = read_json(deepsurv_tuning_config_path)

# ------------------------------
# 5) Build consolidated audit rows
# ------------------------------
audit_rows = []

# ---- linear discrete-time tuned
audit_rows.append({
    "model_id": "linear_tuned",
    "display_name": "Linear Discrete-Time Hazard (Tuned)",
    "family": "discrete_time_linear",
    "data_level": "person_period_weekly",
    "raw_input_design": "weekly person-period with static + temporal-behavioral covariates",
    "numeric_imputation": safe_first(linear_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(linear_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(linear_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(linear_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none (class_weight=None)",
    "n_input_features_raw": safe_first(linear_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(linear_preproc_summary, "n_features_after_transform"),
    "validation_unit": "enrollment",
    "validation_strategy": "GroupShuffleSplit on enrollment_id",
    "validation_fraction": 0.20,
    "selection_metric": linear_tuning_config.get("selection_metric", "not_reported"),
    "search_space": stringify_search_space(linear_tuning_config.get("search_space")),
    "n_tuning_candidates": int(linear_tuning_results.shape[0]),
    "early_stopping_used": "no",
    "early_stopping_patience": "not_applicable",
    "complexity_control": "penalty type + inverse regularization strength C",
    "best_candidate_summary": stringify_search_space(linear_tuning_config.get("best_candidate")),
    "preprocessing_note": linear_preproc_config.get("linear_positioning_note", "not_reported"),
    "tuning_note": linear_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

# ---- neural discrete-time tuned
audit_rows.append({
    "model_id": "neural_tuned",
    "display_name": "Neural Discrete-Time Survival (Tuned)",
    "family": "discrete_time_neural",
    "data_level": "person_period_weekly",
    "raw_input_design": "weekly person-period with same conceptual feature set as linear baseline",
    "numeric_imputation": safe_first(neural_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(neural_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(neural_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(neural_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none (unweighted BCEWithLogitsLoss)",
    "n_input_features_raw": safe_first(neural_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(neural_preproc_summary, "n_features_after_transform"),
    "validation_unit": "enrollment",
    "validation_strategy": "train/validation split from distinct enrollment_id values",
    "validation_fraction": neural_tuning_config.get("validation_fraction", 0.10),
    "selection_metric": neural_tuning_config.get("selection_criterion", "lowest_validation_loss"),
    "search_space": stringify_search_space({
        "hidden_dims": [[64, 32], [128, 64]],
        "dropout": [0.10, 0.30],
        "learning_rate": [1e-3, 5e-4],
        "weight_decay": [1e-5, 1e-4],
    }),
    "n_tuning_candidates": int(neural_tuning_results.shape[0]),
    "early_stopping_used": "yes",
    "early_stopping_patience": neural_tuning_config.get("early_stopping_patience", "not_reported"),
    "complexity_control": "network width/depth grid + dropout + weight decay + early stopping + max epochs",
    "best_candidate_summary": stringify_search_space(neural_tuning_config.get("best_candidate")),
    "preprocessing_note": safe_first(neural_preproc_summary, "comparability_note", default="not_reported"),
    "tuning_note": neural_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

# ---- cox tuned
audit_rows.append({
    "model_id": "cox_tuned",
    "display_name": "Cox Comparable (Tuned)",
    "family": "continuous_time_cox",
    "data_level": "enrollment_early_window",
    "raw_input_design": "static covariates + early-window summaries (4 weeks)",
    "numeric_imputation": safe_first(cox_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(cox_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(cox_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(cox_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none",
    "n_input_features_raw": safe_first(cox_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(cox_preproc_summary, "n_features_after_transform"),
    "validation_unit": "enrollment",
    "validation_strategy": "train/validation split on enrollment_id with event stratification when possible",
    "validation_fraction": 0.20,
    "selection_metric": cox_tuning_config.get("selection_metric", "val_neg_partial_log_likelihood"),
    "search_space": stringify_search_space({
        "penalizer_grid": cox_tuning_config.get("penalizer_grid"),
        "l1_ratio_grid": cox_tuning_config.get("l1_ratio_grid"),
    }),
    "n_tuning_candidates": int(cox_tuning_results.shape[0]),
    "early_stopping_used": "no",
    "early_stopping_patience": "not_applicable",
    "complexity_control": "penalizer + l1_ratio regularization grid",
    "best_candidate_summary": stringify_search_space(cox_tuning_config.get("best_candidate")),
    "preprocessing_note": cox_preproc_config.get("cox_positioning_note", "not_reported"),
    "tuning_note": cox_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

# ---- deepsurv tuned
audit_rows.append({
    "model_id": "deepsurv_tuned",
    "display_name": "DeepSurv (Tuned)",
    "family": "continuous_time_deepsurv",
    "data_level": "enrollment_early_window",
    "raw_input_design": "static covariates + early-window summaries (4 weeks)",
    "numeric_imputation": safe_first(deepsurv_preproc_summary, "numeric_imputation"),
    "categorical_imputation": safe_first(deepsurv_preproc_summary, "categorical_imputation"),
    "categorical_encoding": safe_first(deepsurv_preproc_summary, "categorical_encoding"),
    "numeric_scaling": safe_first(deepsurv_preproc_summary, "numeric_scaling"),
    "fit_on_train_only": "yes",
    "class_imbalance_handling": "none",
    "n_input_features_raw": safe_first(deepsurv_preproc_summary, "n_input_features_raw"),
    "n_features_after_transform": safe_first(deepsurv_preproc_summary, "n_features_after_transform"),
    "validation_unit": "row_within_training_set",
    "validation_strategy": "random internal validation fraction on training rows",
    "validation_fraction": deepsurv_tuning_config.get("validation_fraction", 0.20),
    "selection_metric": deepsurv_tuning_config.get("selection_metric", "best_val_loss"),
    "search_space": stringify_search_space({
        "hidden_dims_grid": deepsurv_tuning_config.get("hidden_dims_grid"),
        "dropout_grid": deepsurv_tuning_config.get("dropout_grid"),
        "learning_rate_grid": deepsurv_tuning_config.get("learning_rate_grid"),
        "weight_decay_grid": deepsurv_tuning_config.get("weight_decay_grid"),
        "batch_norm": deepsurv_tuning_config.get("batch_norm"),
        "epochs": deepsurv_tuning_config.get("epochs"),
    }),
    "n_tuning_candidates": int(deepsurv_tuning_results.shape[0]),
    "early_stopping_used": "yes",
    "early_stopping_patience": deepsurv_tuning_config.get("patience", "not_reported"),
    "complexity_control": "network architecture grid + dropout + weight decay + early stopping + best epoch refit",
    "best_candidate_summary": stringify_search_space(deepsurv_tuning_config.get("best_candidate")),
    "preprocessing_note": deepsurv_preproc_config.get("comparability_note", "not_reported"),
    "tuning_note": deepsurv_tuning_config.get("benchmark_positioning_note", "not_reported"),
})

preprocessing_tuning_audit_df = pd.DataFrame(audit_rows)

# ------------------------------
# 6) Appendix-oriented compact version
# ------------------------------
appendix_audit_df = preprocessing_tuning_audit_df[
    [
        "display_name",
        "family",
        "data_level",
        "numeric_imputation",
        "categorical_imputation",
        "categorical_encoding",
        "numeric_scaling",
        "fit_on_train_only",
        "class_imbalance_handling",
        "validation_unit",
        "validation_strategy",
        "validation_fraction",
        "selection_metric",
        "n_tuning_candidates",
        "early_stopping_used",
        "early_stopping_patience",
        "complexity_control",
    ]
].copy()

appendix_audit_df = appendix_audit_df.rename(columns={
    "display_name": "model",
    "data_level": "input_level",
    "numeric_imputation": "num_impute",
    "categorical_imputation": "cat_impute",
    "categorical_encoding": "cat_encoding",
    "numeric_scaling": "num_scaling",
    "fit_on_train_only": "fit_train_only",
    "class_imbalance_handling": "imbalance",
    "validation_unit": "val_unit",
    "validation_strategy": "val_strategy",
    "validation_fraction": "val_frac",
    "selection_metric": "select_metric",
    "n_tuning_candidates": "n_candidates",
    "early_stopping_used": "early_stop",
    "early_stopping_patience": "patience",
    "complexity_control": "complexity_control",
})

# ------------------------------
# 7) Save outputs
# ------------------------------
audit_path = TABLES_DIR / "table_preprocessing_and_tuning_audit.csv"
appendix_audit_path = TABLES_DIR / "table_appendix_preprocessing_and_tuning_audit_compact.csv"
summary_json_path = METADATA_DIR / "preprocessing_and_tuning_audit_summary.json"

preprocessing_tuning_audit_df.to_csv(audit_path, index=False)
appendix_audit_df.to_csv(appendix_audit_path, index=False)

save_json(
    {
        "n_models": int(preprocessing_tuning_audit_df.shape[0]),
        "model_ids": preprocessing_tuning_audit_df["model_id"].tolist(),
        "appendix_ready_table": str(appendix_audit_path),
        "full_audit_table": str(audit_path),
        "notes": [
            "All preprocessing transformations were fit on training data only.",
            "Discrete-time linear and Cox tuning do not use early stopping.",
            "Neural discrete-time tuning uses enrollment-level internal validation.",
            "DeepSurv tuning uses an internal validation fraction on training rows.",
        ],
    },
    summary_json_path,
)

# ------------------------------
# 8) Output for feedback
# ------------------------------
print("\nFull preprocessing and tuning audit:")
display(preprocessing_tuning_audit_df)

print("\nAppendix-oriented compact audit:")
display(appendix_audit_df)

print("\nSaved:")
print("-", audit_path.resolve())
print("-", appendix_audit_path.resolve())
print("-", summary_json_path.resolve())


# F7 — Bootstrap Uncertainty Audit for Tuned Benchmark Models (Revised v4)

In [ ]:
# ==============================================================
# F7 — Bootstrap Uncertainty Audit for Tuned Benchmark Models (Revised v4)
# --------------------------------------------------------------
# Purpose:
#   Quantify uncertainty for the tuned benchmark hierarchy using
#   enrollment-level bootstrap resampling on the held-out test set.
#
# Methodological note:
#   This step does NOT retrain models during each bootstrap draw.
#   It rebuilds the tuned models' test-set survival predictions once,
#   then performs enrollment-level resampling on the held-out test
#   enrollments to estimate uncertainty in:
#     - IBS
#     - time-dependent concordance
#     - Brier@10, Brier@20, Brier@30
#
#   It also computes rank-stability summaries to assess whether the
#   benchmark ordering is robust enough to justify expressions such as
#   "stable hierarchy".
#
# Important correction in v4:
#   Bootstrap samples are materialized with synthetic unique ids
#   (boot_{iter}_{j}) so that surv_df columns remain unique and
#   perfectly aligned with the corresponding truth table.
# ==============================================================

print("\n" + "=" * 70)
print("F7 — Bootstrap Uncertainty Audit for Tuned Benchmark Models (Revised v4)")
print("=" * 70)
print("Methodological note: this step rebuilds tuned-model survival predictions")
print("on the held-out test set and performs enrollment-level bootstrap")
print("resampling to quantify uncertainty in the benchmark hierarchy.")
print("No model is retrained inside bootstrap iterations.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "HORIZONS_WEEKS", "RANDOM_SEED"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import scipy
import torch
import torch.nn as nn

try:
    from pycox.evaluation import EvalSurv
    from pycox.models import CoxPH
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    import torchtuples as tt
    TORCHTUPLES_AVAILABLE = True
except Exception:
    TORCHTUPLES_AVAILABLE = False

try:
    from lifelines import CoxPHFitter
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P31.6.")
if not TORCHTUPLES_AVAILABLE:
    raise ImportError("torchtuples is required for P31.6.")
if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P31.6.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"
try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Config
# ------------------------------
BOOTSTRAP_CONFIG = {
    "n_bootstrap": 200,
    "ci_alpha": 0.05,
    "metrics": ["ibs", "c_index_td"] + [f"brier_h{h}" for h in HORIZONS_WEEKS],
    "max_horizon_for_ibs": int(max(HORIZONS_WEEKS)),
    "resampling_unit": "enrollment",
    "random_seed": int(RANDOM_SEED),
    "note": (
        "Enrollment-level bootstrap on the held-out test set using fixed tuned-model predictions."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
    "sanity_tolerance_abs": 0.02,
}

N_BOOT = BOOTSTRAP_CONFIG["n_bootstrap"]
ALPHA = BOOTSTRAP_CONFIG["ci_alpha"]
LOW_Q = 100 * (ALPHA / 2)
HIGH_Q = 100 * (1 - ALPHA / 2)
SANITY_TOL = BOOTSTRAP_CONFIG["sanity_tolerance_abs"]

# ------------------------------
# 4) Paths
# ------------------------------
DATA_DIR = OUTPUT_DIR / "data"
MODELS_DIR = OUTPUT_DIR / "models"

COX_TRAIN_PATH = DATA_DIR / "enrollment_cox_ready_train.csv"
COX_TEST_PATH = DATA_DIR / "enrollment_cox_ready_test.csv"
COX_MODEL_PATH = MODELS_DIR / "cox_early_window_tuned.joblib"
COX_PREPROC_PATH = MODELS_DIR / "cox_preprocessor.joblib"
COX_STABILITY_PATH = TABLES_DIR / "table_cox_tuned_stability_notes.csv"
COX_PRIMARY_METRICS_PATH = TABLES_DIR / "table_cox_tuned_primary_metrics.csv"
COX_BRIER_PATH = TABLES_DIR / "table_cox_tuned_brier_by_horizon.csv"

DEEPSURV_TRAIN_PATH = DATA_DIR / "enrollment_deepsurv_ready_train.csv"
DEEPSURV_TEST_PATH = DATA_DIR / "enrollment_deepsurv_ready_test.csv"
DEEPSURV_MODEL_PATH = MODELS_DIR / "deepsurv_tuned.pt"
DEEPSURV_PREPROC_PATH = MODELS_DIR / "deepsurv_preprocessor.joblib"
DEEPSURV_CONFIG_PATH = METADATA_DIR / "deepsurv_tuned_model_config.json"
DEEPSURV_PRIMARY_METRICS_PATH = TABLES_DIR / "table_deepsurv_tuned_primary_metrics.csv"
DEEPSURV_BRIER_PATH = TABLES_DIR / "table_deepsurv_tuned_brier_by_horizon.csv"

LINEAR_TEST_PATH = DATA_DIR / "pp_linear_hazard_ready_test.csv"
LINEAR_TRAIN_PATH = DATA_DIR / "pp_linear_hazard_ready_train.csv"
LINEAR_MODEL_PATH = MODELS_DIR / "linear_discrete_time_hazard_tuned.joblib"
LINEAR_PREPROC_PATH = MODELS_DIR / "linear_discrete_time_preprocessor.joblib"
LINEAR_PRIMARY_METRICS_PATH = TABLES_DIR / "table_linear_tuned_primary_metrics.csv"
LINEAR_BRIER_PATH = TABLES_DIR / "table_linear_tuned_brier_by_horizon.csv"

NEURAL_TEST_PATH = DATA_DIR / "pp_neural_hazard_ready_test.csv"
NEURAL_TRAIN_PATH = DATA_DIR / "pp_neural_hazard_ready_train.csv"
NEURAL_MODEL_PATH = MODELS_DIR / "neural_discrete_time_survival_tuned.pt"
NEURAL_PREPROC_PATH = MODELS_DIR / "neural_discrete_time_preprocessor.joblib"
NEURAL_CONFIG_PATH = METADATA_DIR / "neural_tuned_model_config.json"
NEURAL_PRIMARY_METRICS_PATH = TABLES_DIR / "table_neural_tuned_primary_metrics.csv"
NEURAL_BRIER_PATH = TABLES_DIR / "table_neural_tuned_brier_by_horizon.csv"

required_paths = [
    COX_TRAIN_PATH, COX_TEST_PATH, COX_MODEL_PATH, COX_PREPROC_PATH, COX_STABILITY_PATH,
    COX_PRIMARY_METRICS_PATH, COX_BRIER_PATH,
    DEEPSURV_TRAIN_PATH, DEEPSURV_TEST_PATH, DEEPSURV_MODEL_PATH, DEEPSURV_PREPROC_PATH,
    DEEPSURV_CONFIG_PATH, DEEPSURV_PRIMARY_METRICS_PATH, DEEPSURV_BRIER_PATH,
    LINEAR_TRAIN_PATH, LINEAR_TEST_PATH, LINEAR_MODEL_PATH, LINEAR_PREPROC_PATH,
    LINEAR_PRIMARY_METRICS_PATH, LINEAR_BRIER_PATH,
    NEURAL_TRAIN_PATH, NEURAL_TEST_PATH, NEURAL_MODEL_PATH, NEURAL_PREPROC_PATH,
    NEURAL_CONFIG_PATH, NEURAL_PRIMARY_METRICS_PATH, NEURAL_BRIER_PATH,
]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required files for P31.6:\n- " + "\n- ".join(missing_paths))

# ------------------------------
# 5) Helpers
# ------------------------------
AUX_ENROLLMENT = [
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "duration", "duration_raw", "used_zero_week_fallback_for_censoring",
    "split", "time_for_split", "time_bucket", "event_time_bucket_label"
]
TARGET_ENROLLMENT = ["event"]

AUX_DISCRETE = [
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "event_observed", "t_event_week", "t_final_week",
    "used_zero_week_fallback_for_censoring", "split",
    "time_for_split", "time_bucket", "event_time_bucket_label"
]
TARGET_DISCRETE = ["event_t"]


### Funcao: load_csv

Definicao isolada de load_csv para reutilizacao nas etapas seguintes.


In [ ]:

def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)


### Funcao: read_json

Definicao isolada de read_json para reutilizacao nas etapas seguintes.


In [ ]:

def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


### Funcao: safe_save_json

Definicao isolada de safe_save_json para reutilizacao nas etapas seguintes.


In [ ]:

def safe_save_json(obj: dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


### Funcao: get_feature_cols

Definicao isolada de get_feature_cols para reutilizacao nas etapas seguintes.


In [ ]:

def get_feature_cols(df: pd.DataFrame, aux_cols: list[str], target_cols: list[str]) -> list[str]:
    excluded = set(aux_cols + target_cols)
    return [c for c in df.columns if c not in excluded]


### Funcao: build_truth_from_discrete

Definicao isolada de build_truth_from_discrete para reutilizacao nas etapas seguintes.


In [ ]:

def build_truth_from_discrete(df: pd.DataFrame) -> pd.DataFrame:
    event_col = "event_observed" if "event_observed" in df.columns else "event_t"
    duration_col = "time_for_split" if "time_for_split" in df.columns else "week"
    truth = (
        df.groupby("enrollment_id", as_index=False)
        .agg(
            event=(event_col, "max"),
            duration=(duration_col, "max"),
        )
    )
    truth["event"] = truth["event"].astype(int)
    truth["duration"] = truth["duration"].astype(int)
    return truth


### Funcao: align_truth_to_surv_df

Definicao isolada de align_truth_to_surv_df para reutilizacao nas etapas seguintes.


In [ ]:

def align_truth_to_surv_df(truth_df: pd.DataFrame, surv_df: pd.DataFrame) -> pd.DataFrame:
    truth_idx = truth_df.set_index("enrollment_id")
    missing = [eid for eid in surv_df.columns.tolist() if eid not in truth_idx.index]
    if missing:
        raise KeyError(f"Missing enrollment_ids in truth_df alignment: {missing[:10]}")
    aligned = truth_idx.loc[list(surv_df.columns)].reset_index()
    return aligned


### Funcao: eval_surv_metrics_from_surv_df

Definicao isolada de eval_surv_metrics_from_surv_df para reutilizacao nas etapas seguintes.


In [ ]:

def eval_surv_metrics_from_surv_df(surv_df: pd.DataFrame, truth_df: pd.DataFrame, horizons: list[int]) -> dict:
    truth_aligned = align_truth_to_surv_df(truth_df, surv_df)

    durations = truth_aligned["duration"].astype(int).to_numpy()
    events = truth_aligned["event"].astype(int).to_numpy()

    ev = EvalSurv(
        surv=surv_df,
        durations=durations,
        events=events,
        censor_surv="km",
    )

    out = {}

    try:
        out["ibs"] = float(ev.integrated_brier_score(np.arange(1, int(max(horizons)) + 1)))
    except Exception:
        out["ibs"] = np.nan

    try:
        out["c_index_td"] = float(ev.concordance_td())
    except Exception:
        out["c_index_td"] = np.nan

    try:
        brier_h = ev.brier_score(np.array(horizons, dtype=int))
        for h, v in zip(brier_h.index.astype(int), brier_h.values.astype(float)):
            out[f"brier_h{int(h)}"] = float(v)
    except Exception:
        for h in horizons:
            out[f"brier_h{int(h)}"] = np.nan

    return out


### Funcao: bootstrap_eval_surv_metrics

Definicao isolada de bootstrap_eval_surv_metrics para reutilizacao nas etapas seguintes.


In [ ]:

def bootstrap_eval_surv_metrics(surv_df: pd.DataFrame, truth_df: pd.DataFrame, model_label: str, horizons: list[int], n_boot: int, rng: np.random.Generator) -> pd.DataFrame:
    truth_aligned = align_truth_to_surv_df(truth_df, surv_df)
    enrollment_ids = truth_aligned["enrollment_id"].tolist()
    n = len(enrollment_ids)
    rows = []

    # base lookup once
    truth_lookup = truth_aligned.set_index("enrollment_id")
    surv_lookup = surv_df.copy()

    for b in range(1, n_boot + 1):
        sampled_ids = rng.choice(enrollment_ids, size=n, replace=True).tolist()
        boot_ids = [f"boot_{b}_{j}" for j in range(n)]

        # bootstrap truth with unique synthetic ids
        truth_sample = truth_lookup.loc[sampled_ids].reset_index()
        truth_sample["enrollment_id"] = boot_ids

        # bootstrap survival with same unique synthetic ids
        surv_sample = surv_lookup.loc[:, sampled_ids].copy()
        surv_sample.columns = boot_ids
        surv_sample.columns.name = "enrollment_id"

        metrics = eval_surv_metrics_from_surv_df(surv_sample, truth_sample, horizons)

        row = {"model": model_label, "bootstrap_iter": b}
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows)


### Funcao: summarize_bootstrap

Definicao isolada de summarize_bootstrap para reutilizacao nas etapas seguintes.


In [ ]:

def summarize_bootstrap(boot_df: pd.DataFrame, point_row: dict, metric_cols: list[str], model_label: str) -> pd.DataFrame:
    rows = []
    for metric in metric_cols:
        vals = boot_df[metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
        if len(vals) == 0:
            rows.append({
                "model": model_label,
                "metric": metric,
                "point_estimate": point_row.get(metric, np.nan),
                "bootstrap_mean": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "n_successful_bootstrap": 0,
            })
        else:
            rows.append({
                "model": model_label,
                "metric": metric,
                "point_estimate": point_row.get(metric, np.nan),
                "bootstrap_mean": float(np.mean(vals)),
                "ci_lower": float(np.percentile(vals, LOW_Q)),
                "ci_upper": float(np.percentile(vals, HIGH_Q)),
                "n_successful_bootstrap": int(len(vals)),
            })
    return pd.DataFrame(rows)


### Funcao: build_rank_stability

Definicao isolada de build_rank_stability para reutilizacao nas etapas seguintes.


In [ ]:

def build_rank_stability(boot_long: pd.DataFrame, metric: str, higher_is_better: bool) -> pd.DataFrame:
    rows = []
    metric_df = boot_long[["model", "bootstrap_iter", metric]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    for boot_iter, g in metric_df.groupby("bootstrap_iter"):
        g = g.copy()
        g["rank"] = g[metric].rank(method="min", ascending=not higher_is_better)
        for _, r in g.iterrows():
            rows.append({
                "bootstrap_iter": int(boot_iter),
                "model": r["model"],
                "metric": metric,
                "rank": int(r["rank"]),
            })
    rank_df = pd.DataFrame(rows)
    if rank_df.empty:
        return pd.DataFrame(columns=["model", "metric", "rank", "frequency", "proportion"])

    out = (
        rank_df.groupby(["model", "metric", "rank"])
        .size()
        .reset_index(name="frequency")
    )
    total_per_model_metric = out.groupby(["model", "metric"])["frequency"].transform("sum")
    out["proportion"] = out["frequency"] / total_per_model_metric
    return out.sort_values(["metric", "rank", "model"]).reset_index(drop=True)


### Funcao: build_pairwise_win_probability

Definicao isolada de build_pairwise_win_probability para reutilizacao nas etapas seguintes.


In [ ]:

def build_pairwise_win_probability(boot_long: pd.DataFrame, metric: str, higher_is_better: bool) -> pd.DataFrame:
    models = sorted(boot_long["model"].unique().tolist())
    rows = []
    for i, m1 in enumerate(models):
        for m2 in models[i+1:]:
            tmp1 = boot_long[boot_long["model"] == m1][["bootstrap_iter", metric]].rename(columns={metric: "v1"})
            tmp2 = boot_long[boot_long["model"] == m2][["bootstrap_iter", metric]].rename(columns={metric: "v2"})
            merged = tmp1.merge(tmp2, on="bootstrap_iter", how="inner")
            merged = merged.replace([np.inf, -np.inf], np.nan).dropna()
            if merged.empty:
                prob_m1_better = np.nan
                prob_m2_better = np.nan
            else:
                if higher_is_better:
                    prob_m1_better = float((merged["v1"] > merged["v2"]).mean())
                    prob_m2_better = float((merged["v2"] > merged["v1"]).mean())
                else:
                    prob_m1_better = float((merged["v1"] < merged["v2"]).mean())
                    prob_m2_better = float((merged["v2"] < merged["v1"]).mean())
            rows.append({
                "metric": metric,
                "model_a": m1,
                "model_b": m2,
                "prob_a_better_than_b": prob_m1_better,
                "prob_b_better_than_a": prob_m2_better,
                "n_bootstrap_pairs": int(merged.shape[0]) if not merged.empty else 0,
            })
    return pd.DataFrame(rows)


### Funcao: load_frozen_metrics

Definicao isolada de load_frozen_metrics para reutilizacao nas etapas seguintes.


In [ ]:

def load_frozen_metrics(primary_path: Path, brier_path: Path, model_name: str) -> dict:
    primary = pd.read_csv(primary_path)
    brier = pd.read_csv(brier_path)

    out = {"model": model_name}
    if "metric_name" in primary.columns:
        ibs_row = primary[primary["metric_name"].isin(["ibs", "IBS"])]
        cidx_row = primary[primary["metric_name"].isin(["c_index", "c_index_td", "C-index"])]
        out["ibs"] = float(ibs_row["metric_value"].iloc[0]) if ibs_row.shape[0] > 0 else np.nan
        out["c_index_td"] = float(cidx_row["metric_value"].iloc[0]) if cidx_row.shape[0] > 0 else np.nan
    else:
        out["ibs"] = np.nan
        out["c_index_td"] = np.nan

    if "horizon_week" in brier.columns:
        for h in HORIZONS_WEEKS:
            row = brier[brier["horizon_week"] == h]
            out[f"brier_h{h}"] = float(row["metric_value"].iloc[0]) if row.shape[0] > 0 else np.nan
    else:
        for h in HORIZONS_WEEKS:
            out[f"brier_h{h}"] = np.nan

    return out


### Classe: TunedHazardMLP

Definicao isolada de TunedHazardMLP para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 6) Model builders / predictors
# ------------------------------
class TunedHazardMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


### Funcao: predict_proba_in_batches_torch

Definicao isolada de predict_proba_in_batches_torch para reutilizacao nas etapas seguintes.


In [ ]:

def predict_proba_in_batches_torch(model, X_np: np.ndarray, batch_size: int = 4096, device: str = "cpu") -> np.ndarray:
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, X_np.shape[0], batch_size):
            xb = torch.from_numpy(X_np[start:start+batch_size].astype(np.float32)).to(device)
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy().reshape(-1)
            probs.append(p)
    return np.concatenate(probs, axis=0)


### Funcao: rebuild_linear_surv

Definicao isolada de rebuild_linear_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_linear_surv():
    test_df = load_csv(LINEAR_TEST_PATH)
    model = joblib.load(LINEAR_MODEL_PATH)
    preproc = joblib.load(LINEAR_PREPROC_PATH)

    feature_cols = get_feature_cols(test_df, AUX_DISCRETE, TARGET_DISCRETE)
    X_test = preproc.transform(test_df[feature_cols])
    if hasattr(X_test, "toarray"):
        X_test = X_test.toarray()

    pred_hazard = np.clip(model.predict_proba(X_test)[:, 1], 1e-8, 1 - 1e-8)

    test_pred_df = test_df.copy()
    test_pred_df["pred_hazard"] = pred_hazard
    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )
    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)
    surv_wide.columns.name = "enrollment_id"

    truth = build_truth_from_discrete(test_df)
    return surv_wide, truth


### Funcao: rebuild_neural_surv

Definicao isolada de rebuild_neural_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_neural_surv():
    test_df = load_csv(NEURAL_TEST_PATH)
    preproc = joblib.load(NEURAL_PREPROC_PATH)
    cfg = read_json(NEURAL_CONFIG_PATH)
    best = cfg["best_candidate"]

    feature_cols = get_feature_cols(test_df, AUX_DISCRETE, TARGET_DISCRETE)
    X_test = preproc.transform(test_df[feature_cols])
    if hasattr(X_test, "toarray"):
        X_test = X_test.toarray()

    input_dim = int(X_test.shape[1])
    hidden_dims = best["hidden_dims"]
    dropout = float(best["dropout"])

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = TunedHazardMLP(input_dim=input_dim, hidden_dims=hidden_dims, dropout=dropout).to(device)
    state_dict = torch.load(NEURAL_MODEL_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    pred_hazard = np.clip(
        predict_proba_in_batches_torch(model, np.asarray(X_test), device=device),
        1e-8, 1 - 1e-8
    )

    test_pred_df = test_df.copy()
    test_pred_df["pred_hazard"] = pred_hazard
    test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
    test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
        lambda s: (1.0 - s).cumprod()
    )

    surv_wide = (
        test_pred_df[["enrollment_id", "week", "pred_survival"]]
        .drop_duplicates(subset=["enrollment_id", "week"])
        .pivot(index="week", columns="enrollment_id", values="pred_survival")
        .sort_index()
    )
    max_week_test = int(test_pred_df["week"].max())
    full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
    surv_wide = surv_wide.reindex(full_week_index).ffill().fillna(1.0)
    surv_wide.columns.name = "enrollment_id"

    truth = build_truth_from_discrete(test_df)
    return surv_wide, truth


### Funcao: rebuild_cox_surv

Definicao isolada de rebuild_cox_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_cox_surv():
    test_df = load_csv(COX_TEST_PATH)
    model = joblib.load(COX_MODEL_PATH)
    preproc = joblib.load(COX_PREPROC_PATH)
    stability = pd.read_csv(COX_STABILITY_PATH)

    feature_cols = get_feature_cols(test_df, AUX_ENROLLMENT, TARGET_ENROLLMENT)
    X_test = preproc.transform(test_df[feature_cols])
    feature_names_out = list(preproc.get_feature_names_out())
    X_test_df = pd.DataFrame(X_test, columns=feature_names_out)

    dropped_cols = []
    if "dropped_zero_variance_features" in stability.columns:
        raw = str(stability.loc[0, "dropped_zero_variance_features"])
        if raw.strip() != "" and raw.strip().lower() != "nan":
            dropped_cols = [x.strip() for x in raw.split(";") if x.strip()]

    keep_cols = [c for c in X_test_df.columns if c not in dropped_cols]
    X_test_df = X_test_df[keep_cols].copy()

    times = np.arange(0, int(test_df["duration"].max()) + 1, dtype=int)
    pred_surv = model.predict_survival_function(X_test_df, times=times)

    surv_df = pred_surv.copy()
    surv_df.columns = test_df["enrollment_id"].tolist()
    surv_df.columns.name = "enrollment_id"
    surv_df.index = surv_df.index.astype(int)

    truth = test_df[["enrollment_id", "event", "duration"]].copy()
    truth["event"] = truth["event"].astype(int)
    truth["duration"] = truth["duration"].astype(int)
    return surv_df, truth


### Funcao: rebuild_deepsurv_surv

Definicao isolada de rebuild_deepsurv_surv para reutilizacao nas etapas seguintes.


In [ ]:

def rebuild_deepsurv_surv():
    train_df = load_csv(DEEPSURV_TRAIN_PATH)
    test_df = load_csv(DEEPSURV_TEST_PATH)
    preproc = joblib.load(DEEPSURV_PREPROC_PATH)
    cfg = read_json(DEEPSURV_CONFIG_PATH)
    best = cfg["best_candidate"]

    feature_cols = get_feature_cols(test_df, AUX_ENROLLMENT, TARGET_ENROLLMENT)

    X_train = preproc.transform(train_df[feature_cols]).astype(np.float32)
    X_test = preproc.transform(test_df[feature_cols]).astype(np.float32)

    y_train = (
        train_df["duration"].astype(np.float32).to_numpy(),
        train_df["event"].astype(np.float32).to_numpy(),
    )

    net = tt.practical.MLPVanilla(
        in_features=X_train.shape[1],
        num_nodes=best["hidden_dims"],
        out_features=1,
        batch_norm=True,
        dropout=float(best["dropout"]),
        output_bias=False,
    )

    model = CoxPH(net, tt.optim.AdamW)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    state_dict = torch.load(DEEPSURV_MODEL_PATH, map_location=device)
    model.net.load_state_dict(state_dict)
    model.compute_baseline_hazards(input=X_train, target=y_train)

    surv_df = model.predict_surv_df(X_test)
    surv_df.columns = test_df["enrollment_id"].tolist()
    surv_df.columns.name = "enrollment_id"
    surv_df.index = surv_df.index.astype(float)

    truth = test_df[["enrollment_id", "event", "duration"]].copy()
    truth["event"] = truth["event"].astype(int)
    truth["duration"] = truth["duration"].astype(int)
    return surv_df, truth


In [ ]:

# ------------------------------
# 7) Rebuild tuned-model survival predictions
# ------------------------------
print("\nRebuilding tuned-model survival objects on the held-out test set...")

model_surv_objects = {}

surv_linear, truth_linear = rebuild_linear_surv()
model_surv_objects["Linear Discrete-Time Hazard (Tuned)"] = {
    "surv_df": surv_linear,
    "truth_df": truth_linear,
    "frozen_metrics": load_frozen_metrics(LINEAR_PRIMARY_METRICS_PATH, LINEAR_BRIER_PATH, "Linear Discrete-Time Hazard (Tuned)"),
}

surv_neural, truth_neural = rebuild_neural_surv()
model_surv_objects["Neural Discrete-Time Survival (Tuned)"] = {
    "surv_df": surv_neural,
    "truth_df": truth_neural,
    "frozen_metrics": load_frozen_metrics(NEURAL_PRIMARY_METRICS_PATH, NEURAL_BRIER_PATH, "Neural Discrete-Time Survival (Tuned)"),
}

surv_cox, truth_cox = rebuild_cox_surv()
model_surv_objects["Cox Comparable (Tuned)"] = {
    "surv_df": surv_cox,
    "truth_df": truth_cox,
    "frozen_metrics": load_frozen_metrics(COX_PRIMARY_METRICS_PATH, COX_BRIER_PATH, "Cox Comparable (Tuned)"),
}

surv_deepsurv, truth_deepsurv = rebuild_deepsurv_surv()
model_surv_objects["DeepSurv (Tuned)"] = {
    "surv_df": surv_deepsurv,
    "truth_df": truth_deepsurv,
    "frozen_metrics": load_frozen_metrics(DEEPSURV_PRIMARY_METRICS_PATH, DEEPSURV_BRIER_PATH, "DeepSurv (Tuned)"),
}

print("Done.")

# ------------------------------
# 8) Point estimates from rebuilt survival objects
# ------------------------------
point_rows = []
for model_name, obj in model_surv_objects.items():
    metrics = eval_surv_metrics_from_surv_df(obj["surv_df"], obj["truth_df"], HORIZONS_WEEKS)
    row = {"model": model_name}
    row.update(metrics)
    point_rows.append(row)

point_estimates_df = pd.DataFrame(point_rows)

print("\nPoint estimates recomputed from rebuilt survival objects:")
display(point_estimates_df)

# ------------------------------
# 9) Sanity audit against frozen metrics
# ------------------------------
frozen_rows = []
for model_name, obj in model_surv_objects.items():
    frozen_rows.append(obj["frozen_metrics"])
frozen_estimates_df = pd.DataFrame(frozen_rows)

sanity_rows = []
metric_cols = ["ibs", "c_index_td"] + [f"brier_h{h}" for h in HORIZONS_WEEKS]
for _, point_row in point_estimates_df.iterrows():
    model_name = point_row["model"]
    frozen_row = frozen_estimates_df[frozen_estimates_df["model"] == model_name].iloc[0]
    for metric in metric_cols:
        rebuilt_val = point_row.get(metric, np.nan)
        frozen_val = frozen_row.get(metric, np.nan)
        abs_diff = abs(rebuilt_val - frozen_val) if pd.notna(rebuilt_val) and pd.notna(frozen_val) else np.nan
        sanity_rows.append({
            "model": model_name,
            "metric": metric,
            "rebuilt_value": rebuilt_val,
            "frozen_value": frozen_val,
            "abs_diff": abs_diff,
            "within_tolerance": bool(abs_diff <= SANITY_TOL) if pd.notna(abs_diff) else False,
        })

sanity_audit_df = pd.DataFrame(sanity_rows)

print("\nSanity audit against frozen tuned-model metrics:")
display(sanity_audit_df)

bad_rows = sanity_audit_df[sanity_audit_df["within_tolerance"] == False].copy()
if bad_rows.shape[0] > 0:
    raise RuntimeError(
        "Rebuilt survival objects failed sanity comparison against frozen metrics. "
        "Do not proceed to bootstrap until the alignment/rebuild issue is resolved."
    )

# ------------------------------
# 10) Bootstrap uncertainty
# ------------------------------
rng = np.random.default_rng(BOOTSTRAP_CONFIG["random_seed"])

bootstrap_all = []
for model_name, obj in model_surv_objects.items():
    print(f"Bootstrapping: {model_name}")
    boot_df = bootstrap_eval_surv_metrics(
        surv_df=obj["surv_df"],
        truth_df=obj["truth_df"],
        model_label=model_name,
        horizons=HORIZONS_WEEKS,
        n_boot=N_BOOT,
        rng=rng,
    )
    bootstrap_all.append(boot_df)

bootstrap_long_df = pd.concat(bootstrap_all, ignore_index=True)

# ------------------------------
# 11) Summaries
# ------------------------------
summary_frames = []
for _, point_row in point_estimates_df.iterrows():
    model_name = point_row["model"]
    boot_df = bootstrap_long_df[bootstrap_long_df["model"] == model_name].copy()
    tmp = summarize_bootstrap(
        boot_df=boot_df,
        point_row=point_row.to_dict(),
        metric_cols=metric_cols,
        model_label=model_name,
    )
    summary_frames.append(tmp)

bootstrap_summary_df = pd.concat(summary_frames, ignore_index=True)

appendix_compact_df = bootstrap_summary_df.copy()
appendix_compact_df["ci_95"] = appendix_compact_df.apply(
    lambda r: (
        f"[{r['ci_lower']:.3f}, {r['ci_upper']:.3f}]"
        if pd.notna(r["ci_lower"]) and pd.notna(r["ci_upper"])
        else "NA"
    ),
    axis=1,
)
appendix_compact_df = appendix_compact_df[[
    "model", "metric", "point_estimate", "ci_95", "n_successful_bootstrap"
]].copy()

# ------------------------------
# 12) Rank stability / pairwise stability
# ------------------------------
rank_ibs_df = build_rank_stability(bootstrap_long_df, metric="ibs", higher_is_better=False)
rank_cidx_df = build_rank_stability(bootstrap_long_df, metric="c_index_td", higher_is_better=True)
rank_stability_df = pd.concat([rank_ibs_df, rank_cidx_df], ignore_index=True)

pairwise_ibs_df = build_pairwise_win_probability(bootstrap_long_df, metric="ibs", higher_is_better=False)
pairwise_cidx_df = build_pairwise_win_probability(bootstrap_long_df, metric="c_index_td", higher_is_better=True)
pairwise_stability_df = pd.concat([pairwise_ibs_df, pairwise_cidx_df], ignore_index=True)

# ------------------------------
# 13) Save artifacts
# ------------------------------
point_estimates_path = TABLES_DIR / "table_bootstrap_point_estimates_tuned_models.csv"
bootstrap_long_path = TABLES_DIR / "table_bootstrap_metrics_long_tuned_models.csv"
bootstrap_summary_path = TABLES_DIR / "table_bootstrap_uncertainty_summary_tuned_models.csv"
appendix_compact_path = TABLES_DIR / "table_appendix_bootstrap_uncertainty_compact.csv"
rank_stability_path = TABLES_DIR / "table_bootstrap_rank_stability_tuned_models.csv"
pairwise_stability_path = TABLES_DIR / "table_bootstrap_pairwise_stability_tuned_models.csv"
sanity_audit_path = TABLES_DIR / "table_bootstrap_rebuild_sanity_audit.csv"
config_path = METADATA_DIR / "bootstrap_uncertainty_audit_config.json"

point_estimates_df.to_csv(point_estimates_path, index=False)
bootstrap_long_df.to_csv(bootstrap_long_path, index=False)
bootstrap_summary_df.to_csv(bootstrap_summary_path, index=False)
appendix_compact_df.to_csv(appendix_compact_path, index=False)
rank_stability_df.to_csv(rank_stability_path, index=False)
pairwise_stability_df.to_csv(pairwise_stability_path, index=False)
sanity_audit_df.to_csv(sanity_audit_path, index=False)

safe_save_json(BOOTSTRAP_CONFIG, config_path)

# ------------------------------
# 14) Output for feedback
# ------------------------------
print("\nBootstrap summary (full):")
display(bootstrap_summary_df)

print("\nAppendix-oriented compact bootstrap summary:")
display(appendix_compact_df)

print("\nRank stability:")
display(rank_stability_df)

print("\nPairwise stability:")
display(pairwise_stability_df)

print("\nSaved:")
print("-", point_estimates_path.resolve())
print("-", bootstrap_long_path.resolve())
print("-", bootstrap_summary_path.resolve())
print("-", appendix_compact_path.resolve())
print("-", rank_stability_path.resolve())
print("-", pairwise_stability_path.resolve())
print("-", sanity_audit_path.resolve())
print("-", config_path.resolve())


# F8 — Proportional Hazards Audit for the Comparable Cox Model

In [ ]:
# ==============================================================
# F8 — Proportional Hazards Audit for the Comparable Cox Model
# (Revised v2)
# --------------------------------------------------------------
# Purpose:
#   Perform a formal proportional-hazards (PH) diagnostic for the
#   tuned comparable Cox model and export paper-ready audit artifacts.
# ==============================================================

print("\n" + "=" * 70)
print("F8 — Proportional Hazards Audit for the Comparable Cox Model (Revised v2)")
print("=" * 70)

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
    )

import warnings
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

try:
    from lifelines import CoxPHFitter
    from lifelines.statistics import proportional_hazard_test
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P31.7.")

# ------------------------------
# 2) Resolve raw/base Cox train dataframe
# ------------------------------
cox_train_base_df = None

candidate_df_names = [
    "train_df_cox",
    "cox_train_df",
    "enrollment_cox_ready_train",
    "df_cox_train",
]

for name in candidate_df_names:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame) and obj.shape[0] > 0:
            cox_train_base_df = obj.copy()
            print(f"Resolved Cox base dataframe from in-memory object: {name}")
            break

if cox_train_base_df is None and "con" in globals():
    candidate_queries = [
        "SELECT * FROM enrollment_cox_ready_train",
        "SELECT * FROM cox_ready_train",
        "SELECT * FROM train_df_cox",
    ]
    for q in candidate_queries:
        try:
            tmp = con.execute(q).fetchdf()
            if isinstance(tmp, pd.DataFrame) and tmp.shape[0] > 0:
                cox_train_base_df = tmp.copy()
                print(f"Resolved Cox base dataframe from SQL query: {q}")
                break
        except Exception:
            pass

if cox_train_base_df is None:
    raise NameError(
        "Could not resolve a Cox training dataframe "
        "(train_df_cox / cox_train_df / enrollment_cox_ready_train)."
    )

# ------------------------------
# 3) Resolve duration and event columns
# ------------------------------
duration_col = None
event_col = None

duration_candidates = ["duration", "time", "time_to_event", "T"]
event_candidates = ["event", "event_observed", "E"]

for c in duration_candidates:
    if c in cox_train_base_df.columns:
        duration_col = c
        break

for c in event_candidates:
    if c in cox_train_base_df.columns:
        event_col = c
        break

if duration_col is None or event_col is None:
    raise ValueError(
        f"Could not identify duration/event columns. "
        f"duration_col={duration_col}, event_col={event_col}"
    )

# ------------------------------
# 4) Try to resolve already-transformed Cox matrix
# ------------------------------
X_cox_train = None
cox_feature_names = None

X_candidates = [
    "X_train_cox",
    "X_cox_train",
    "X_train_comparable_cox",
    "X_train_continuous_time_cox",
]
feature_name_candidates = [
    "cox_feature_names_out",
    "feature_names_out_cox",
    "cox_selected_features",
    "cox_model_feature_cols",
    "cox_covariate_columns",
]

for name in X_candidates:
    if name in globals():
        X_candidate = globals()[name]
        try:
            if hasattr(X_candidate, "shape") and X_candidate.shape[0] == cox_train_base_df.shape[0]:
                X_cox_train = X_candidate
                print(f"Resolved transformed Cox matrix from: {name}")
                break
        except Exception:
            pass

for name in feature_name_candidates:
    if name in globals():
        f_candidate = globals()[name]
        if isinstance(f_candidate, (list, tuple, np.ndarray)) and len(f_candidate) > 0:
            cox_feature_names = [str(x) for x in f_candidate]
            print(f"Resolved Cox feature names from: {name}")
            break

# ------------------------------
# 5) If needed, resolve Cox preprocessor and transform
# ------------------------------
cox_preprocessor_obj = None
preprocessor_candidates = [
    "cox_preprocessor",
    "preprocessor_cox",
    "comparable_cox_preprocessor",
]

for name in preprocessor_candidates:
    if name in globals():
        cox_preprocessor_obj = globals()[name]
        print(f"Resolved Cox preprocessor from: {name}")
        break

exclude_cols = {
    duration_col, event_col,
    "enrollment_id", "id_student", "code_module", "code_presentation",
    "student_id", "module_presentation_id"
}

if X_cox_train is None:
    if cox_preprocessor_obj is None:
        raise ValueError(
            "Could not resolve a transformed Cox matrix and no Cox preprocessor "
            "was found to rebuild it."
        )

    raw_feature_cols = [c for c in cox_train_base_df.columns if c not in exclude_cols]
    if len(raw_feature_cols) == 0:
        raise ValueError("No raw feature columns available to rebuild Cox transformed matrix.")

    X_raw = cox_train_base_df[raw_feature_cols].copy()
    X_cox_train = cox_preprocessor_obj.transform(X_raw)
    print("Rebuilt transformed Cox matrix using the resolved Cox preprocessor.")

    if cox_feature_names is None:
        if hasattr(cox_preprocessor_obj, "get_feature_names_out"):
            cox_feature_names = [str(x) for x in cox_preprocessor_obj.get_feature_names_out()]
            print("Recovered transformed feature names from preprocessor.get_feature_names_out().")

# ------------------------------
# 6) Build numeric Cox audit dataframe
# ------------------------------
if hasattr(X_cox_train, "toarray"):
    X_cox_dense = X_cox_train.toarray()
else:
    X_cox_dense = np.asarray(X_cox_train)

if X_cox_dense.ndim != 2:
    raise ValueError(f"Transformed Cox matrix must be 2D. Got shape={X_cox_dense.shape}")

n_rows, n_cols = X_cox_dense.shape

if cox_feature_names is None:
    cox_feature_names = [f"x_{i}" for i in range(n_cols)]
elif len(cox_feature_names) != n_cols:
    print(
        f"Warning: feature-name length mismatch "
        f"(len={len(cox_feature_names)} vs n_cols={n_cols}). "
        f"Falling back to generic names."
    )
    cox_feature_names = [f"x_{i}" for i in range(n_cols)]

cox_numeric_df = pd.DataFrame(X_cox_dense, columns=cox_feature_names, index=cox_train_base_df.index)

cox_model_df = pd.concat(
    [
        cox_train_base_df[[duration_col, event_col]].copy(),
        cox_numeric_df
    ],
    axis=1
)

cox_model_df = cox_model_df.replace([np.inf, -np.inf], np.nan)

n_before = cox_model_df.shape[0]
cox_model_df = cox_model_df.dropna(axis=0).copy()
n_after = cox_model_df.shape[0]

if n_after == 0:
    raise ValueError("After dropping NaN/Inf rows, no rows remain for PH audit.")

cox_model_df[duration_col] = pd.to_numeric(cox_model_df[duration_col], errors="raise")
cox_model_df[event_col] = pd.to_numeric(cox_model_df[event_col], errors="raise").astype(int)

covariate_cols = [c for c in cox_model_df.columns if c not in [duration_col, event_col]]

# drop constant columns
non_constant_covariates = []
dropped_constant_covariates = []

for c in covariate_cols:
    if cox_model_df[c].nunique(dropna=True) <= 1:
        dropped_constant_covariates.append(c)
    else:
        non_constant_covariates.append(c)

covariate_cols = non_constant_covariates
cox_model_df = cox_model_df[[duration_col, event_col] + covariate_cols].copy()

if len(covariate_cols) == 0:
    raise ValueError("No non-constant numeric Cox covariates remained for PH audit.")

print(f"Rows before cleaning: {n_before:,}")
print(f"Rows after cleaning:  {n_after:,}")
print(f"Covariates tested:    {len(covariate_cols):,}")
if dropped_constant_covariates:
    print(f"Dropped constant covariates: {len(dropped_constant_covariates)}")

# ------------------------------
# 7) Resolve tuned Cox penalization settings if available
# ------------------------------
penalizer_value = 0.001
l1_ratio_value = 0.0

candidate_config_objs = [
    "COX_TUNING_CONFIG",
    "cox_tuning_config",
    "cox_best_config",
    "cox_model_config",
]

for name in candidate_config_objs:
    if name in globals():
        cfg = globals()[name]
        try:
            if isinstance(cfg, dict):
                best = cfg.get("best_candidate", cfg)
                if "penalizer" in best:
                    penalizer_value = float(best["penalizer"])
                if "l1_ratio" in best:
                    l1_ratio_value = float(best["l1_ratio"])
        except Exception:
            pass

print(f"Audit Cox refit with penalizer={penalizer_value}, l1_ratio={l1_ratio_value}")

# ------------------------------
# 8) Fit audit Cox model
# ------------------------------
cox_audit_model = CoxPHFitter(
    penalizer=penalizer_value,
    l1_ratio=l1_ratio_value,
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cox_audit_model.fit(
        cox_model_df,
        duration_col=duration_col,
        event_col=event_col,
        show_progress=False
    )

# ------------------------------
# 9) Formal PH test
# ------------------------------
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ph_test_rank = proportional_hazard_test(
        cox_audit_model,
        cox_model_df,
        time_transform="rank"
    )

ph_summary = ph_test_rank.summary.copy().reset_index()
if "index" in ph_summary.columns:
    ph_summary = ph_summary.rename(columns={"index": "covariate"})

rename_map = {}
for col in ph_summary.columns:
    cl = col.lower()
    if cl in ["p", "p_value"]:
        rename_map[col] = "p_value"
    elif cl in ["test_statistic", "test statistic", "chi2"]:
        rename_map[col] = "test_statistic"
ph_summary = ph_summary.rename(columns=rename_map)

if "p_value" not in ph_summary.columns:
    p_candidates = [c for c in ph_summary.columns if "p" in c.lower()]
    if p_candidates:
        ph_summary = ph_summary.rename(columns={p_candidates[0]: "p_value"})

if "test_statistic" not in ph_summary.columns:
    stat_candidates = [c for c in ph_summary.columns if "test" in c.lower() or "chi" in c.lower()]
    if stat_candidates:
        ph_summary = ph_summary.rename(columns={stat_candidates[0]: "test_statistic"})

ph_summary["p_value"] = pd.to_numeric(ph_summary["p_value"], errors="coerce")
ph_summary["test_statistic"] = pd.to_numeric(ph_summary["test_statistic"], errors="coerce")

alpha = 0.05
ph_summary["ph_flag"] = np.where(
    ph_summary["p_value"] < alpha,
    "possible_violation",
    "no_material_evidence"
)
ph_summary["ph_flag_binary"] = np.where(ph_summary["p_value"] < alpha, "yes", "no")

ph_audit_df = (
    ph_summary[["covariate", "test_statistic", "p_value", "ph_flag", "ph_flag_binary"]]
    .sort_values(["p_value", "covariate"], ascending=[True, True])
    .reset_index(drop=True)
)

# ------------------------------
# 10) Global summary
# ------------------------------
n_tested = int(ph_audit_df.shape[0])
n_flagged = int((ph_audit_df["p_value"] < alpha).sum())
top_flagged = ph_audit_df.loc[ph_audit_df["p_value"] < alpha, "covariate"].head(5).tolist()

if n_flagged == 0:
    global_classification = "A_no_material_evidence"
    global_interpretation = (
        "No material evidence of proportional-hazards violation was detected "
        "for the comparable Cox benchmark at the chosen threshold."
    )
elif n_flagged <= max(2, int(0.15 * n_tested)):
    global_classification = "B_localized_departures"
    global_interpretation = (
        "The comparable Cox benchmark showed some localized departures from "
        "proportional hazards, but not a pattern severe enough to prevent its "
        "use as the methodological anchor of the comparable continuous-time family."
    )
else:
    global_classification = "C_broad_departure"
    global_interpretation = (
        "The comparable Cox benchmark showed broad evidence of proportional-hazards "
        "departure and should therefore be interpreted as an approximate comparable "
        "benchmark rather than a fully assumption-clean Cox specification."
    )

global_summary_df = pd.DataFrame([{
    "model": "Cox Comparable (Tuned)",
    "duration_col": duration_col,
    "event_col": event_col,
    "n_rows_used": int(cox_model_df.shape[0]),
    "n_covariates_tested": n_tested,
    "alpha": alpha,
    "n_covariates_flagged": n_flagged,
    "share_covariates_flagged": float(n_flagged / n_tested) if n_tested > 0 else np.nan,
    "top_flagged_covariates": "; ".join(top_flagged) if top_flagged else "none",
    "global_classification": global_classification,
    "global_interpretation": global_interpretation,
    "audit_note": (
        "Formal proportional-hazards audit for the comparable Cox branch. "
        "DeepSurv shares the Cox-type ranking structure but was not subjected "
        "to an identical classical PH diagnostic."
    ),
}])

# ------------------------------
# 11) Optional figure
# ------------------------------
figure_created = False
figure_note = "not_created"

figures_dir = OUTPUT_DIR / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

appendix_fig_dir = figures_dir / "paper_appendix"
appendix_fig_dir.mkdir(parents=True, exist_ok=True)

fig_path_png = appendix_fig_dir / "fig_appendix_cox_ph_diagnostics.png"
fig_path_pdf = appendix_fig_dir / "fig_appendix_cox_ph_diagnostics.pdf"

if MATPLOTLIB_AVAILABLE:
    try:
        plot_df = ph_audit_df.nsmallest(min(8, ph_audit_df.shape[0]), "p_value").copy()
        plot_df = plot_df.sort_values("p_value", ascending=False)

        fig, ax = plt.subplots(figsize=(8.5, 4.8))
        ax.barh(plot_df["covariate"], plot_df["p_value"])
        ax.axvline(alpha, linestyle="--")
        ax.set_xlabel("PH test p-value")
        ax.set_ylabel("Covariate")
        ax.set_title("Comparable Cox PH diagnostic (smallest p-values)")
        plt.tight_layout()

        fig.savefig(fig_path_png, dpi=300, bbox_inches="tight")
        fig.savefig(fig_path_pdf, bbox_inches="tight")
        plt.close(fig)

        figure_created = True
        figure_note = "top_smallest_p_values_barh"
    except Exception as e:
        figure_note = f"figure_failed: {str(e)}"

# ------------------------------
# 12) Save artifacts
# ------------------------------
paper_appendix_tables_dir = TABLES_DIR / "paper_appendix"
paper_appendix_tables_dir.mkdir(parents=True, exist_ok=True)

audit_path_main = TABLES_DIR / "table_appendix_cox_ph_audit.csv"
global_summary_path_main = TABLES_DIR / "table_appendix_cox_ph_global_summary.csv"

audit_path_paper = paper_appendix_tables_dir / "table_paper_appendix_cox_ph_audit.csv"
global_summary_path_paper = paper_appendix_tables_dir / "table_paper_appendix_cox_ph_global_summary.csv"

metadata_path = METADATA_DIR / "cox_ph_audit_summary.json"

ph_audit_df.to_csv(audit_path_main, index=False)
global_summary_df.to_csv(global_summary_path_main, index=False)
ph_audit_df.to_csv(audit_path_paper, index=False)
global_summary_df.to_csv(global_summary_path_paper, index=False)

metadata_payload = {
    "step_id": "P31.7",
    "model": "Cox Comparable (Tuned)",
    "duration_col": duration_col,
    "event_col": event_col,
    "n_rows_before_cleaning": int(n_before),
    "n_rows_after_cleaning": int(n_after),
    "n_rows_used": int(cox_model_df.shape[0]),
    "n_covariates_tested": n_tested,
    "n_covariates_flagged": n_flagged,
    "alpha": alpha,
    "global_classification": global_classification,
    "global_interpretation": global_interpretation,
    "top_flagged_covariates": top_flagged,
    "dropped_constant_covariates": dropped_constant_covariates,
    "penalizer": penalizer_value,
    "l1_ratio": l1_ratio_value,
    "figure_created": figure_created,
    "figure_note": figure_note,
    "deep_surv_note": (
        "DeepSurv shares the Cox-type ranking structure but was not subjected "
        "to an identical classical proportional-hazards diagnostic."
    ),
}
save_json(metadata_payload, metadata_path)

# ------------------------------
# 13) Output
# ------------------------------
print("\nGlobal PH audit summary:")
display(global_summary_df)

print("\nPH audit by covariate:")
display(ph_audit_df)

if figure_created:
    print("\nPH diagnostic figure created:")
    print("-", fig_path_png.resolve())
    print("-", fig_path_pdf.resolve())
else:
    print("\nPH diagnostic figure not created.")
    print("Reason:", figure_note)

print("\nSaved artifacts:")
print("-", audit_path_main.resolve())
print("-", global_summary_path_main.resolve())
print("-", audit_path_paper.resolve())
print("-", global_summary_path_paper.resolve())
print("-", metadata_path.resolve())